# 🛰️ Workshop · IA Multi-Agéntica Centralizada para Redes de Acceso
**Curso Redes de Acceso · Bloque de 90 minutos · Parte práctica**

> Basado en P. González y L. Velasco, *"Demonstration of Centralized Multi-Agentic AI for Telecom Network Operation"*, GCO–UPC, IEEE, 2026 (y sus Refs. [1] JOCN 2025, [2] JOCN 2024, [3] JOCN 2024).

### 🧭 Ruta del workshop
| Fase | Dónde | Tiempo |
|---|---|---|
| **1 · Teoría** | Presentación HTML (ya revisada) | 0:00 – 0:20 |
| **2 · Experimentación guiada** | Este notebook · **Estaciones 0 → 4** | 0:20 – 1:05 |
| **3 · Exploración interactiva** | Este notebook · **Estación 5** | 1:05 – 1:25 |
| Cierre | Conclusiones | 1:25 – 1:30 |

### 🚉 Estaciones
| # | Estación | ⏱️ | Lo que veremos | Visualización |
|---|---|---|---|---|
| 0 | Preparación | 5' | Entorno listo, modo simulado/real | Diagnóstico |
| 1 | La red y su control SDN | 8' | Mininet/OVS, OpenFlow, Ryu, aislamiento H1↔H2 | Topología + tabla de flujos |
| 2 | Herramientas deterministas | 8' | Inventario, PCE, servidor **MCP** | Ruta del PCE + catálogo MCP |
| 3 | El "cerebro" | 8' | **ChromaDB** (zero-shot) + **Mistral 7B** | Barras de similitud + plan JSON |
| 4 | Agentes **A2A** y flujo de 8 pasos | 12' | Handoff, Figuras 3, 4 y 5 del paper | Diagrama de secuencia real |
| 5 | Exploración interactiva | 20' | Escenarios libres + extensiones de acceso | Consola interactiva |

### 📐 Cómo está organizada cada estación
🎯 **Objetivo** · 📖 **Concepto en 1 minuto** (con la diapositiva de referencia) · ⚙️ **Código de soporte** plegado (solo ejecutar) · ▶️ **Ejecutar y observar** · 💬 **Pregunta rápida** para comentar entre todos.

```text
 Operador ─①─► ORQUESTADOR ─②─► LLM local (Mistral 7B) + ChromaDB
    ▲                │ ▲
    └──────⑧─────────┘ │⑦
             ③ A2A (Handoff)
                     ▼ │
              IPConnectionAgent ◄─MCP─► Servidor MCP ─④─► PCE + Inventario
                                                   └─⑤─► Ryu SDN (NBI) ─⑥ OpenFlow─► OVS: H1─R1─R2─R3─H2
```
👥 **Avanzamos todos juntos, celda por celda**, con `Shift + Enter`.

---
## 🟦⬜⬜⬜⬜⬜ Estación 0 · Preparación del entorno · ⏱️ 5 min
🎯 **Objetivo:** dejar el entorno listo y elegir el modo de ejecución.

📖 **Concepto en 1 minuto** *(Diapositiva 14)*: cada componente del paper es un **servicio independiente** (HTTP) que correrá en segundo plano dentro de este notebook. Con `SIMULATION_MODE = True` un emulador reproduce Mininet + OVS + Ryu en cualquier PC; con `USE_OLLAMA = "auto"` se usa **Mistral 7B** local si Ollama está disponible (si no, un LLM emulado con la misma interfaz).

In [ ]:
# ============================ CONFIGURACIÓN DEL WORKSHOP ============================
import os, sys, platform, shutil

SIMULATION_MODE = True          # True: emulador (cualquier PC) | False: Mininet + OVS + Ryu reales (Linux + sudo)
USE_OLLAMA = "auto"             # "auto": Mistral 7B si Ollama responde | True: obligatorio | False: LLM emulado
OLLAMA_HOST = "http://127.0.0.1:11434"
OLLAMA_MODEL = "mistral:7b"     # Mistral 7B (7.3e9 parámetros); 'mistral:latest' también sirve
AGENT_LLM_TOOL_MAPPING = False  # True: el agente también consulta al LLM por cada paso (más realista, más lento)
INSTALL_PACKAGES = True

# Overrides por variables de entorno (ejecución automatizada)
if os.environ.get("WORKSHOP_SIMULATION_MODE"):
    SIMULATION_MODE = os.environ["WORKSHOP_SIMULATION_MODE"].lower() in ("1", "true", "yes")
if os.environ.get("WORKSHOP_USE_OLLAMA"):
    USE_OLLAMA = {"true": True, "false": False}.get(os.environ["WORKSHOP_USE_OLLAMA"].lower(), "auto")

if not SIMULATION_MODE:
    faltantes = [t for t in ("mn", "ovs-vsctl", "ryu-manager") if shutil.which(t) is None]
    if platform.system() != "Linux" or faltantes:
        print(f"⚠️  Modo real no disponible (SO={platform.system()}, faltan={faltantes}) -> SIMULATION_MODE=True")
        SIMULATION_MODE = True

PROMPT = "Establish an IP connection from H1 to H2 with 1GB bandwidth"   # la intención del paper
print(f"Python {sys.version.split()[0]} | SIMULATION_MODE={SIMULATION_MODE} | USE_OLLAMA={USE_OLLAMA}")

In [ ]:
# ⚙️ Instalación de paquetes faltantes
import importlib.util, subprocess
REQUIRED = {"chromadb": "chromadb", "ollama": "ollama", "fastmcp": "fastmcp", "networkx": "networkx",
            "fastapi": "fastapi", "uvicorn": "uvicorn", "requests": "requests", "pydantic": "pydantic",
            "ipywidgets": "ipywidgets", "matplotlib": "matplotlib"}
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing and INSTALL_PACKAGES:
    print("Instalando:", missing)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], capture_output=True, text=True)
    print(r.stderr[-1500:] if r.returncode else "")
    importlib.invalidate_caches()
print("  ".join(f"{'✅' if importlib.util.find_spec(m) else '❌'} {m}" for m in REQUIRED))

> ⚙️ **Código de soporte (celdas plegadas).** Las siguientes celdas escriben/arrancan los componentes de esta estación: `workshop_common.py (configuración, captura tipo Wireshark, servidores en segundo plano)` · `workshop_viz.py (gráficos)`.
> **Solo ejecútelas** (Shift+Enter); no es necesario leerlas en clase. Puede desplegarlas para ver el detalle completo.

In [ ]:
%%writefile workshop_common.py
"""
workshop_common.py
==================
Utilidades compartidas por TODOS los componentes del workshop:

  * Configuración global persistida en ``workshop_config.json``.
  * ``TrafficCapture``: captura de tráfico "estilo Wireshark" (Figura 3 del paper).
  * ``http_call``: cliente HTTP instrumentado que registra cada request/response.
  * ``start_server`` / ``stop_server``: lanzan apps FastAPI con uvicorn en hilos
    *daemon*, de modo que ninguna celda bloquee el kernel de Jupyter.
"""
from __future__ import annotations

import copy
import json
import os
import socket
import threading
import time
from typing import Any, Callable

import requests

CONFIG_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "workshop_config.json")

DEFAULT_CONFIG: dict[str, Any] = {
    "SIMULATION_MODE": True,          # True: Mininet/Ryu emulados | False: Mininet + Ryu reales
    "HOST": "127.0.0.1",
    "PORTS": {
        "orchestrator": 8000,         # Orquestador (POST /prompt)
        "llm": 8001,                  # Servicio LLM + ChromaDB (POST /plan)
        "agent": 8002,                # IPConnectionAgent (A2A, POST /execute_plan)
        "pce": 8003,                  # Path Computation Element (POST /compute_route)
        "optical_agent": 8004,        # OpticalP2MPAgent (Ejercicio 2)
        "mcp_http": 8005,             # Servidor MCP standalone (opcional, transporte HTTP)
        "sdn": 8080,                  # NBI REST del controlador SDN (Ryu wsapi o emulador)
        "mininet_ctl": 9000,          # Canal de control del script Mininet (modo real)
    },
    "USE_OLLAMA": "auto",             # "auto" | True | False
    "OLLAMA_HOST": "http://127.0.0.1:11434",
    "OLLAMA_MODEL": "mistral:7b",
    "EMBEDDING_BACKEND": "auto",      # "auto" | "ollama" | "chroma-default" | "hashing"
    "OLLAMA_EMBED_MODEL": "nomic-embed-text",
    "SDN_BACKEND": "ryu",             # "ryu" (Ryu/emulador) | "tfs" (ETSI TeraFlowSDN)
    "TFS_URL": "http://127.0.0.1:80",
    "TFS_CONTEXT": "admin",
    "TFS_DRY_RUN": True,
}

CONFIG: dict[str, Any] = {}


def load_config() -> dict[str, Any]:
    """Carga la configuración por defecto y la sobreescribe con workshop_config.json."""
    cfg = copy.deepcopy(DEFAULT_CONFIG)
    if os.path.exists(CONFIG_PATH):
        with open(CONFIG_PATH, encoding="utf-8") as fh:
            user_cfg = json.load(fh)
        ports = {**cfg["PORTS"], **user_cfg.get("PORTS", {})}
        cfg.update(user_cfg)
        cfg["PORTS"] = ports
    CONFIG.clear()
    CONFIG.update(cfg)
    return CONFIG


def save_config(**overrides: Any) -> dict[str, Any]:
    """Actualiza la configuración en memoria y la persiste en disco."""
    if not CONFIG:
        load_config()
    for key, value in overrides.items():
        if key == "PORTS":
            CONFIG["PORTS"].update(value)
        else:
            CONFIG[key] = value
    with open(CONFIG_PATH, "w", encoding="utf-8") as fh:
        json.dump(CONFIG, fh, indent=2)
    return CONFIG


def url(service: str) -> str:
    """URL base de un servicio del workshop, p.ej. url('pce') -> http://127.0.0.1:8003"""
    return f"http://{CONFIG['HOST']}:{CONFIG['PORTS'][service]}"


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, ensure_ascii=False)


# =============================================================================
# Prompt de sistema del planificador (compartido con el precalentamiento del LLM)
# =============================================================================
EXTRACTION_SYSTEM_PROMPT = (
    "You are the planning component of a telecom network operations system. "
    "Your ONLY task is to extract parameter values from the operator request. "
    "Do not compute routes and do not invent values. "
    "Network node identifiers are short uppercase codes such as H1, H2, R1, R3, ONU2. "
    "Bandwidths must be expressed in Gb/s as a plain number (e.g. '1GB' -> 1, '500 Mbps' -> 0.5). "
    "Reply ONLY with a JSON object whose keys are exactly: {keys}. "
    "Parameter meanings: {meanings}"
)
PARAM_MEANINGS = {
    "src_node": "source node of the connection",
    "dst_node": "destination node of the connection",
    "bitrate_gbps": "requested bandwidth in Gb/s (number)",
    "leaf_node": "leaf node (ONU / access node) of the point-to-multipoint tree",
    "num_sc": "number of digital subcarriers (integer)",
}
LLM_OPTIONS = {"temperature": 0, "seed": 42}
LLM_KEEP_ALIVE = "60m"


def extraction_system_prompt(keys) -> str:
    return EXTRACTION_SYSTEM_PROMPT.format(
        keys=", ".join(keys), meanings="; ".join(f"{k}: {PARAM_MEANINGS.get(k, k)}" for k in keys))


def warmup_llm(host: str, model: str, prompt: str,
               keys=("src_node", "dst_node", "bitrate_gbps")) -> threading.Thread:
    """Carga el modelo en Ollama y deja en su caché el prompt de sistema del planificador.

    En CPU, la primera evaluación de ese prompt puede tardar minutos; hacerlo en segundo
    plano al inicio evita esperas cuando el planificador lo use (Estación 3).
    """
    def _run() -> None:
        try:
            requests.post(f"{host.rstrip('/')}/api/chat", timeout=900, json={
                "model": model, "stream": False, "format": "json", "keep_alive": LLM_KEEP_ALIVE,
                "options": LLM_OPTIONS,
                "messages": [{"role": "system", "content": extraction_system_prompt(keys)},
                             {"role": "user", "content": prompt}]})
        except Exception:
            pass
    thread = threading.Thread(target=_run, name="llm-warmup", daemon=True)
    thread.start()
    return thread


load_config()


# =============================================================================
# Captura de tráfico estilo Wireshark
# =============================================================================
class TrafficCapture:
    """Registro ordenado de los mensajes intercambiados entre componentes."""

    def __init__(self) -> None:
        self._lock = threading.Lock()
        self.frames: list[dict[str, Any]] = []
        self._t0 = time.perf_counter()

    def reset(self) -> None:
        with self._lock:
            self.frames = []
            self._t0 = time.perf_counter()

    def record(self, src: str, dst: str, protocol: str, info: str,
               payload: Any = None, length: int | None = None) -> dict[str, Any]:
        with self._lock:
            if payload is None:
                body = ""
            elif isinstance(payload, str):
                body = payload
            else:
                body = json.dumps(payload)
            frame = {
                "no": len(self.frames) + 1,
                "time": time.perf_counter() - self._t0,
                "src": src,
                "dst": dst,
                "protocol": protocol,
                "length": length if length is not None else 66 + len(body.encode("utf-8")),
                "info": info,
                "payload": payload,
            }
            self.frames.append(frame)
            return frame

    def filter(self, protocols: tuple[str, ...] | None = None) -> list[dict[str, Any]]:
        with self._lock:
            frames = list(self.frames)
        if protocols is None:
            return frames
        return [f for f in frames if f["protocol"] in protocols]

    def to_table(self, protocols: tuple[str, ...] | None = ("HTTP", "OpenFlow"),
                 annotate: Callable[[dict[str, Any]], str] | None = None) -> str:
        frames = self.filter(protocols)
        header = f"{'No.':>4}  {'Time':>10}  {'Source':<13} {'Destination':<13} {'Protocol':<9} {'Length':>6}  "
        if annotate:
            header += f"{'Paso':<5} "
        header += "Info"
        lines = [header, "-" * max(len(header), 110)]
        for i, f in enumerate(frames, 1):
            line = (f"{i:>4}  {f['time']:>10.6f}  {f['src']:<13.13} {f['dst']:<13.13} "
                    f"{f['protocol']:<9.9} {f['length']:>6}  ")
            if annotate:
                line += f"{annotate(f):<5} "
            lines.append(line + f["info"])
        return "\n".join(lines)

    def show(self, protocols: tuple[str, ...] | None = ("HTTP", "OpenFlow"),
             annotate: Callable[[dict[str, Any]], str] | None = None) -> None:
        print(self.to_table(protocols, annotate))


CAPTURE = TrafficCapture()


class HTTPCallError(RuntimeError):
    def __init__(self, status: int, detail: Any):
        super().__init__(f"HTTP {status}: {detail}")
        self.status = status
        self.detail = detail


def http_call(src: str, dst: str, method: str, full_url: str, payload: Any = None,
              timeout: float = 600.0) -> Any:
    """Petición HTTP instrumentada: registra request y response en CAPTURE."""
    rest = full_url.split("://", 1)[-1]
    path = rest[rest.find("/"):] if "/" in rest else "/"
    info = f"{method} {path} HTTP/1.1"
    if payload is not None:
        info += "  (application/json)"
    CAPTURE.record(src, dst, "HTTP", info, payload)
    resp = requests.request(method, full_url, json=payload, timeout=timeout)
    try:
        data: Any = resp.json()
    except ValueError:
        data = resp.text
    CAPTURE.record(dst, src, "HTTP", f"HTTP/1.1 {resp.status_code} {resp.reason}  (application/json)", data)
    if resp.status_code >= 400:
        raise HTTPCallError(resp.status_code, data)
    return data


# =============================================================================
# Servidores HTTP en segundo plano (no bloquean el kernel de Jupyter)
# =============================================================================
_SERVERS: dict[str, tuple[Any, threading.Thread, int]] = {}


def port_in_use(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0


def find_free_port(host: str = "127.0.0.1") -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind((host, 0))
        return s.getsockname()[1]


def wait_for_port(host: str, port: int, timeout: float = 15.0) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        if port_in_use(host, port):
            return True
        time.sleep(0.1)
    return False


def start_server(name: str, app: Any, port_key: str) -> str:
    """Lanza ``app`` (FastAPI) en un hilo daemon y devuelve su URL base.

    Si el puerto configurado está ocupado por otro proceso, se elige uno libre
    y se actualiza ``workshop_config.json`` para que el resto de componentes lo use.
    """
    import uvicorn

    class _ThreadedServer(uvicorn.Server):
        def install_signal_handlers(self) -> None:  # compatibilidad uvicorn antiguo
            pass

    stop_server(name)
    host = CONFIG["HOST"]
    port = int(CONFIG["PORTS"][port_key])
    if port_in_use(host, port):
        new_port = find_free_port(host)
        print(f"⚠️  Puerto {port} ocupado: '{name}' usará el puerto {new_port}")
        port = new_port
        save_config(PORTS={port_key: port})
    config = uvicorn.Config(app, host=host, port=port, log_level="warning", lifespan="off")
    server = _ThreadedServer(config)
    thread = threading.Thread(target=server.run, name=f"srv-{name}", daemon=True)
    thread.start()
    deadline = time.time() + 15
    while not server.started and time.time() < deadline:
        time.sleep(0.05)
    if not server.started:
        raise RuntimeError(f"No se pudo iniciar el servidor '{name}' en {host}:{port}")
    _SERVERS[name] = (server, thread, port)
    return f"http://{host}:{port}"


def stop_server(name: str) -> None:
    entry = _SERVERS.pop(name, None)
    if entry is None:
        return
    server, thread, _ = entry
    server.should_exit = True
    thread.join(timeout=5)


def stop_all_servers() -> None:
    for name in list(_SERVERS):
        stop_server(name)


def running_servers() -> dict[str, str]:
    return {name: f"http://{CONFIG['HOST']}:{port}" for name, (_, _, port) in _SERVERS.items()}


In [ ]:
%%writefile workshop_viz.py
"""
workshop_viz.py
===============
Visualizaciones pequeñas (matplotlib) que acompañan cada estación del workshop:

  * plot_network   : topología H1–R1–R2–R3–H2 con estado de enlaces, ruta del PCE,
                     conexión provisionada y política de aislamiento.
  * plot_similarity: similitud coseno (ChromaDB) entre prompts y workflows.
  * plot_sequence  : diagrama de secuencia generado desde la captura real (Fig. 2/3).
"""
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

POS = {"H1": (0, 0), "R1": (1.4, 0), "R2": (2.8, 0), "R3": (4.2, 0), "H2": (5.6, 0)}
ROLES = {"H1": "ONU/CPE\n10.0.0.1", "H2": "BNG/Edge\n10.0.0.2"}
C = {"bg": "#0b1220", "txt": "#e6edf7", "mut": "#9fb0cc", "host": "#9ecae1", "sw": "#fdae6b",
     "up": "#c9d6ea", "standby": "#ba68c8", "degraded": "#e57373", "conn": "#4fc3f7", "pce": "#ffb74d",
     "req": "#4fc3f7", "resp": "#81c784", "of": "#f06292", "mcp": "#ffb74d", "bad": "#e57373"}


def _style(ax, fig):
    fig.patch.set_facecolor(C["bg"])
    ax.set_facecolor(C["bg"])
    ax.axis("off")


def _links(source):
    """Acepta un NetworkInventory (con .graph) o el dict TOPOLOGY de access_topology.py."""
    if hasattr(source, "graph"):
        return [(u, v, d) for u, v, d in source.graph.edges(data=True)]
    return [(l["a"], l["b"], l) for l in source["links"]]


def plot_network(source, title, connection_path=None, computed_path=None, isolated=False):
    """Dibuja la red de acceso/agregación y su estado actual."""
    fig, ax = plt.subplots(figsize=(10, 3.3))
    _style(ax, fig)
    conn_edges = {frozenset(e) for e in zip(connection_path or [], (connection_path or [])[1:])}
    pce_edges = {frozenset(e) for e in zip(computed_path or [], (computed_path or [])[1:])}
    for u, v, d in _links(source):
        (x1, y1), (x2, y2) = POS[u], POS[v]
        residual = d["capacity_gbps"] - d["used_gbps"]
        color = C.get(d["state"], C["up"])
        style = "-" if d["state"] == "up" else "--"
        rad = 0.45 if d["layer"] == "optical" else 0.0
        key = frozenset((u, v))
        if key in conn_edges:
            ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), connectionstyle=f"arc3,rad={rad}", arrowstyle="-",
                                         color=C["conn"], lw=16, alpha=0.55, zorder=1))
        if key in pce_edges:
            ax.add_patch(FancyArrowPatch((x1, y1 - 0.06), (x2, y2 - 0.06), connectionstyle=f"arc3,rad={rad}",
                                         arrowstyle="-", color=C["pce"], lw=2.5, linestyle=":", zorder=3))
        ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), connectionstyle=f"arc3,rad={rad}", arrowstyle="-",
                                     color=color, lw=2, linestyle=style, zorder=2))
        lx, ly = (x1 + x2) / 2, (y1 + y2) / 2 + (0.16 if d["layer"] == "packet" else -1.18)
        label = f"{d['state']} · {residual:g}/{d['capacity_gbps']:g} Gb/s libres"
        if d["layer"] == "optical":
            label = "bypass óptico R1–R3 · " + label
        ax.text(lx, ly, label, color=color, fontsize=8, ha="center")
    for n, (x, y) in POS.items():
        if n.startswith("H"):
            ax.add_patch(plt.Circle((x, y), 0.22, color=C["host"], zorder=4))
            ax.text(x, y - 0.5, ROLES[n], color=C["mut"], fontsize=8, ha="center", va="top")
        else:
            ax.add_patch(plt.Rectangle((x - 0.25, y - 0.17), 0.5, 0.34, color=C["sw"], zorder=4))
        ax.text(x, y, n, color="#04121f", fontsize=10, ha="center", va="center", fontweight="bold", zorder=5)
    if isolated:
        ax.text(2.8, 0.62, "✖  Política de aislamiento activa: H1 y H2 no se alcanzan", color=C["bad"],
                fontsize=11, ha="center", fontweight="bold")
    elif connection_path:
        ax.text(2.8, 0.62, "✔  Conexión H1–H2 activa: " + " → ".join(connection_path), color=C["conn"],
                fontsize=11, ha="center", fontweight="bold")
    legend = [("conexión provisionada (SDN)", C["conn"]), ("ruta calculada (PCE)", C["pce"]),
              ("enlace standby", C["standby"]), ("enlace degradado", C["degraded"])]
    for i, (txt, col) in enumerate(legend):
        ax.text(-0.3 + i * 1.6, -1.68, "■ " + txt, color=col, fontsize=8)
    ax.set_xlim(-0.5, 6.1)
    ax.set_ylim(-1.78, 0.85)
    ax.set_title(title, color=C["txt"], fontsize=12, loc="left")
    plt.tight_layout()
    plt.show()


def plot_similarity(kb, prompts):
    """Barras de similitud coseno entre cada prompt y todos los workflows de ChromaDB."""
    n = len(prompts)
    fig, axes = plt.subplots(1, n, figsize=(3.4 * n, 2.9), sharey=True)
    axes = [axes] if n == 1 else axes
    names = sorted(h["name"] for h in kb.search(prompts[0], k=kb.collection.count()))
    for ax, prompt in zip(axes, prompts):
        hits = {h["name"]: h["similarity"] for h in kb.search(prompt, k=kb.collection.count())}
        best = max(hits, key=hits.get)
        vals = [hits[nm] for nm in names]
        ax.barh(names, vals, color=[C["conn"] if nm == best else "#3a4c70" for nm in names])
        ax.set_facecolor(C["bg"])
        ax.set_xlim(min(0, min(vals)) - 0.05, 1)
        ax.tick_params(colors=C["mut"], labelsize=8)
        for s in ax.spines.values():
            s.set_color("#2a3a5c")
        short = prompt if len(prompt) <= 38 else prompt[:36] + "…"
        ax.set_title(f"«{short}»", color=C["txt"], fontsize=8.5)
    fig.patch.set_facecolor(C["bg"])
    fig.suptitle("ChromaDB · similitud coseno prompt ↔ workflow (azul = workflow recuperado)",
                 color=C["txt"], fontsize=11, x=0.01, ha="left")
    plt.tight_layout()
    plt.show()


LANES = ["User", "Orchestrator", "LLM", "Agent", "PCE", "SDN", "OVS"]
LANE_LABEL = {"User": "Operador", "Orchestrator": "Orquestador", "LLM": "LLM+ChromaDB", "Agent": "Agente (A2A)",
              "PCE": "PCE", "SDN": "Ryu SDN", "OVS": "OVS (Mininet)"}


def plot_sequence(frames, step_fn, title="Diagrama de secuencia generado desde la captura real"):
    """Diagrama de secuencia (HTTP + OpenFlow agregado) a partir de wc.CAPTURE.frames."""
    rows, i = [], 0
    while i < len(frames):
        f = frames[i]
        if f["protocol"] == "OpenFlow":
            j = i
            while j < len(frames) and frames[j]["protocol"] == "OpenFlow":
                j += 1
            n_mod = sum("FLOW_MOD" in x["info"] for x in frames[i:j])
            if n_mod:
                rows.append(("SDN", "OVS", f"{n_mod}× OFPT_FLOW_MOD", C["of"], "6"))
            i = j
            continue
        if f["protocol"] == "HTTP" and f["src"].split()[0] in LANES and f["dst"].split()[0] in LANES:
            resp = f["info"].startswith("HTTP/1.1")
            label = f["info"].replace("  (application/json)", "").replace(" HTTP/1.1", "")
            rows.append((f["src"], f["dst"], label, C["resp"] if resp else C["req"], step_fn(f)))
        i += 1
    fig, ax = plt.subplots(figsize=(11, 0.42 * len(rows) + 1.2))
    _style(ax, fig)
    xs = {lane: k for k, lane in enumerate(LANES)}
    for lane, x in xs.items():
        ax.plot([x, x], [0.4, len(rows) + 0.4], color="#2a3a5c", lw=1.2, zorder=1)
        ax.text(x, 0, LANE_LABEL[lane], color=C["txt"], fontsize=9, ha="center", va="center", fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.35", fc="#121b2e", ec="#4fc3f7"))
    for r, (src, dst, label, color, step) in enumerate(rows, 1):
        x1, x2 = xs[src], xs[dst]
        ax.annotate("", xy=(x2, r), xytext=(x1, r), arrowprops=dict(arrowstyle="-|>", color=color, lw=1.8), zorder=2)
        ax.text((x1 + x2) / 2, r - 0.12, label, color=color, fontsize=8, ha="center", va="bottom")
        if step:
            ax.text(min(x1, x2) - 0.12, r, step, color="#04121f", fontsize=8, ha="right", va="center",
                    fontweight="bold", bbox=dict(boxstyle="circle,pad=0.25", fc=color, ec="none"))
    ax.set_xlim(-0.7, len(LANES) - 0.4)
    ax.set_ylim(len(rows) + 0.7, -0.6)
    ax.set_title(title + "   (círculo = paso de la Figura 2)", color=C["txt"], fontsize=11, loc="left")
    plt.tight_layout()
    plt.show()


In [ ]:
import importlib, json, time, requests

def load_module(name):
    # Importa (o recarga tras un %%writefile) un módulo auxiliar del workshop
    if name in sys.modules and name != "workshop_common":
        return importlib.reload(sys.modules[name])
    return importlib.import_module(name)

wc = load_module("workshop_common")
viz = load_module("workshop_viz")
wc.save_config(SIMULATION_MODE=SIMULATION_MODE, USE_OLLAMA=USE_OLLAMA, OLLAMA_HOST=OLLAMA_HOST,
               OLLAMA_MODEL=OLLAMA_MODEL, SDN_BACKEND="ryu")
try:
    modelos = [m["name"] for m in requests.get(f"{OLLAMA_HOST}/api/tags", timeout=2).json().get("models", [])]
    ollama_txt = "✅ activo · " + (", ".join(modelos) or "sin modelos")
except Exception:
    modelos, ollama_txt = [], "— no responde (se usará el LLM emulado)"

# Precalentamiento: cargar Mistral y su prompt de planificación EN SEGUNDO PLANO (Estaciones 1 y 2)
_mistral = next((m for m in modelos if m == OLLAMA_MODEL or m.split(":")[0] == OLLAMA_MODEL.split(":")[0]), None)
if USE_OLLAMA is not False and _mistral:
    wc.warmup_llm(OLLAMA_HOST, _mistral, PROMPT)
    ollama_txt += f" · 🔥 precalentando {_mistral} en segundo plano"
print(f"Sistema    : {platform.system()} · Python {sys.version.split()[0]}")
print(f"Ollama     : {ollama_txt}")
print(f"Modo       : {'SIMULADO (emulador OVS/Ryu)' if SIMULATION_MODE else 'REAL (Mininet + Ryu)'}")
print(f"Servicios  : {wc.CONFIG['PORTS']}")

---
## 🟦🟦⬜⬜⬜⬜ Estación 1 · La red y su control SDN · ⏱️ 8 min
🎯 **Objetivo:** ver el plano de datos (H1 acceso ↔ H2 borde) y comprobar que, **antes de cualquier intención**, el controlador SDN mantiene ambos nodos **aislados**.

📖 **Concepto en 1 minuto** *(Diapositiva 8)*
* **Mininet + Open vSwitch:** red virtual real en un PC; los switches R1–R3 solo obedecen las **tablas de flujo** que les escribe el controlador.
* **OpenFlow 1.3 (interfaz Sur):** cada regla es *match → acción*. Si no hay regla, el switch pregunta al controlador (`PACKET_IN`); el controlador instala reglas con `FLOW_MOD`.
* **Ryu (`ryu_controller.py`)**: switch L2 + **aislamiento H1↔H2 por defecto** + **NBI REST** (`/service/ip/create | status | delete`) que usará el agente.

```python
# Lo esencial de la app Ryu: si el paquete (ARP/IPv4) va entre un par aislado, se descarta
if self._is_isolated(pkt):
    return                      # -> h1 ping h2 = "Destination Host Unreachable"
```
> En producción, la misma NBI se mapea al controlador *cloud-native* **ETSI TeraFlowSDN** (`POST /tfs-api/context/admin/services`, ver `tfs_adapter.py`).

> ⚙️ **Código de soporte (celdas plegadas).** Las siguientes celdas escriben/arrancan los componentes de esta estación: `access_topology.py` · `ryu_controller.py` · `mininet_topo.py` · `sdn_emulator.py` · `tfs_adapter.py` · `arranque del plano de datos`.
> **Solo ejecútelas** (Shift+Enter); no es necesario leerlas en clase. Puede desplegarlas para ver el detalle completo.

In [ ]:
%%writefile access_topology.py
"""
access_topology.py
==================
Única fuente de verdad de la topología del laboratorio (la usan Mininet, Ryu,
el emulador SDN y el inventario/PCE).

    H1 ──[p1] R1 [p2]──[p1] R2 [p2]──[p1] R3 [p2]── H2
              [p3] ════════ bypass óptico ════════ [p3]   (standby)

  * H1: nodo de acceso (ONU/CPE agregada)       10.0.0.1
  * H2: nodo de borde (BNG / Edge Server / MEC)  10.0.0.2
  * R1-R3: switches OpenFlow 1.3 (OVS) de agregación metro-acceso
  * R1-R3 (óptico): lightpath de *bypass* en standby (Ejercicio 1). No se
    instancia en Mininet; solo existe en el inventario y en el emulador.
"""
from __future__ import annotations

TOPOLOGY: dict = {
    "hosts": {
        "H1": {"name": "h1", "ip": "10.0.0.1", "mac": "00:00:00:00:00:01",
               "switch": "R1", "port": 1, "role": "Acceso: ONU/CPE agregada"},
        "H2": {"name": "h2", "ip": "10.0.0.2", "mac": "00:00:00:00:00:02",
               "switch": "R3", "port": 2, "role": "Borde: BNG / Edge Server (MEC)"},
    },
    "switches": {
        "R1": {"name": "r1", "dpid": 1, "role": "Agregación de acceso (OVS)"},
        "R2": {"name": "r2", "dpid": 2, "role": "Agregación metro (OVS)"},
        "R3": {"name": "r3", "dpid": 3, "role": "Borde metro / PoP (OVS)"},
    },
    "links": [
        {"a": "H1", "b": "R1", "a_port": None, "b_port": 1, "layer": "packet",
         "capacity_gbps": 10.0, "used_gbps": 0.0, "delay_ms": 0.05, "state": "up", "in_mininet": True},
        {"a": "R1", "b": "R2", "a_port": 2, "b_port": 1, "layer": "packet",
         "capacity_gbps": 10.0, "used_gbps": 0.0, "delay_ms": 0.50, "state": "up", "in_mininet": True},
        {"a": "R2", "b": "R3", "a_port": 2, "b_port": 1, "layer": "packet",
         "capacity_gbps": 10.0, "used_gbps": 8.0, "delay_ms": 0.50, "state": "up", "in_mininet": True},
        {"a": "R3", "b": "H2", "a_port": 2, "b_port": None, "layer": "packet",
         "capacity_gbps": 10.0, "used_gbps": 0.0, "delay_ms": 0.05, "state": "up", "in_mininet": True},
        {"a": "R1", "b": "R3", "a_port": 3, "b_port": 3, "layer": "optical",
         "capacity_gbps": 100.0, "used_gbps": 0.0, "delay_ms": 0.30, "state": "standby", "in_mininet": False},
    ],
}


def port_towards(node: str, neighbor: str, topology: dict = TOPOLOGY) -> int:
    """Puerto OpenFlow del switch ``node`` que conduce hacia ``neighbor``."""
    for host_name, host in topology["hosts"].items():
        if neighbor == host_name and host["switch"] == node:
            return host["port"]
    for link in topology["links"]:
        if link["a"] == node and link["b"] == neighbor and link["a_port"] is not None:
            return link["a_port"]
        if link["b"] == node and link["a"] == neighbor and link["b_port"] is not None:
            return link["b_port"]
    raise ValueError(f"No existe enlace {node} -> {neighbor} en la topología")


def default_path(src: str, dst: str, topology: dict = TOPOLOGY) -> list[str]:
    """Camino BFS por los enlaces de paquetes activos (respaldo si no llega ruta del PCE)."""
    adj: dict[str, list[str]] = {}
    for link in topology["links"]:
        if link["state"] == "up":
            adj.setdefault(link["a"], []).append(link["b"])
            adj.setdefault(link["b"], []).append(link["a"])
    frontier, parent = [src], {src: None}
    while frontier:
        nxt = []
        for node in frontier:
            for nb in adj.get(node, []):
                if nb not in parent:
                    parent[nb] = node
                    nxt.append(nb)
        frontier = nxt
    if dst not in parent:
        raise ValueError(f"No hay camino entre {src} y {dst}")
    path, cur = [], dst
    while cur is not None:
        path.append(cur)
        cur = parent[cur]
    return path[::-1]


In [ ]:
%%writefile ryu_controller.py
"""
ryu_controller.py  (a.k.a. ryu_isolation_controller.py en el instructivo)
=========================================================================
Aplicación Ryu (OpenFlow 1.3) usada en la Sección 3 del paper:

  1. Funciona como *switch* de aprendizaje L2 estándar (reglas MAC -> puerto).
  2. POR DEFECTO AÍSLA el tráfico entre H1 y H2: el ARP/IPv4 entre ambos se
     descarta en el controlador, por lo que ``h1 ping h2`` devuelve
     "Destination Host Unreachable".
  3. Expone una API REST (ryu.app.wsgi) que actúa como NBI del controlador:
        POST   /service/ip/create   -> levanta el aislamiento e instala flujos
        GET    /service/ip/status   -> estado de conexiones y aislamiento
        DELETE /service/ip/delete   -> elimina flujos y restablece el aislamiento

Ejecución (Linux, Ryu instalado):
    ryu-manager ryu_controller.py --ofp-tcp-listen-port 6653 --wsapi-port 8080
"""
import json

from ryu.app.wsgi import ControllerBase, WSGIApplication, route
from ryu.base import app_manager
from ryu.controller import ofp_event
from ryu.controller.handler import CONFIG_DISPATCHER, DEAD_DISPATCHER, MAIN_DISPATCHER, set_ev_cls
from ryu.lib.packet import arp, ether_types, ethernet, ipv4, packet
from ryu.ofproto import ofproto_v1_3
from webob import Response

from access_topology import TOPOLOGY, default_path, port_towards

API_INSTANCE = "ip_service_api_app"
LEARNING_COOKIE = 0x1000
SERVICE_COOKIE_BASE = 0xA000
SERVICE_PRIORITY = 100
LEARNING_PRIORITY = 1


class IsolationController(app_manager.RyuApp):
    """Switch L2 con aislamiento H1<->H2 por defecto + NBI REST."""

    OFP_VERSIONS = [ofproto_v1_3.OFP_VERSION]
    _CONTEXTS = {"wsgi": WSGIApplication}

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.mac_to_port = {}
        self.datapaths = {}
        self.connections = {}
        self._next_cookie = SERVICE_COOKIE_BASE
        h1, h2 = TOPOLOGY["hosts"]["H1"]["ip"], TOPOLOGY["hosts"]["H2"]["ip"]
        self.isolated_pairs = {frozenset((h1, h2))}
        kwargs["wsgi"].register(IPServiceRestAPI, {API_INSTANCE: self})

    # ------------------------------------------------------------------ OpenFlow
    @set_ev_cls(ofp_event.EventOFPSwitchFeatures, CONFIG_DISPATCHER)
    def switch_features_handler(self, ev):
        dp = ev.msg.datapath
        ofp, parser = dp.ofproto, dp.ofproto_parser
        self.datapaths[dp.id] = dp
        # Regla table-miss: todo paquete desconocido sube al controlador (PACKET_IN)
        actions = [parser.OFPActionOutput(ofp.OFPP_CONTROLLER, ofp.OFPCML_NO_BUFFER)]
        self.add_flow(dp, 0, parser.OFPMatch(), actions)
        self.logger.info("Switch dpid=%s conectado (OpenFlow 1.3)", dp.id)

    @set_ev_cls(ofp_event.EventOFPStateChange, [MAIN_DISPATCHER, DEAD_DISPATCHER])
    def state_change_handler(self, ev):
        dp = ev.datapath
        if ev.state == MAIN_DISPATCHER:
            self.datapaths[dp.id] = dp
        elif ev.state == DEAD_DISPATCHER:
            self.datapaths.pop(dp.id, None)

    def add_flow(self, dp, priority, match, actions, cookie=0, idle_timeout=0, buffer_id=None):
        """Envía un OFPT_FLOW_MOD (ADD) al switch."""
        ofp, parser = dp.ofproto, dp.ofproto_parser
        inst = [parser.OFPInstructionActions(ofp.OFPIT_APPLY_ACTIONS, actions)]
        kwargs = dict(datapath=dp, cookie=cookie, priority=priority, match=match,
                      instructions=inst, idle_timeout=idle_timeout)
        if buffer_id is not None and buffer_id != ofp.OFP_NO_BUFFER:
            kwargs["buffer_id"] = buffer_id
        dp.send_msg(parser.OFPFlowMod(**kwargs))

    def delete_flows(self, dp, cookie):
        """Envía un OFPT_FLOW_MOD (DELETE) filtrando por cookie."""
        ofp, parser = dp.ofproto, dp.ofproto_parser
        dp.send_msg(parser.OFPFlowMod(
            datapath=dp, cookie=cookie, cookie_mask=0xFFFFFFFFFFFFFFFF,
            table_id=ofp.OFPTT_ALL, command=ofp.OFPFC_DELETE,
            out_port=ofp.OFPP_ANY, out_group=ofp.OFPG_ANY))

    @staticmethod
    def _ip_pair(pkt):
        arp_pkt = pkt.get_protocol(arp.arp)
        if arp_pkt is not None:
            return arp_pkt.src_ip, arp_pkt.dst_ip
        ip_pkt = pkt.get_protocol(ipv4.ipv4)
        if ip_pkt is not None:
            return ip_pkt.src, ip_pkt.dst
        return None

    def _is_isolated(self, pkt):
        pair = self._ip_pair(pkt)
        return pair is not None and frozenset(pair) in self.isolated_pairs

    @set_ev_cls(ofp_event.EventOFPPacketIn, MAIN_DISPATCHER)
    def packet_in_handler(self, ev):
        msg = ev.msg
        dp = msg.datapath
        ofp, parser = dp.ofproto, dp.ofproto_parser
        in_port = msg.match["in_port"]
        pkt = packet.Packet(msg.data)
        eth = pkt.get_protocols(ethernet.ethernet)[0]
        if eth.ethertype in (ether_types.ETH_TYPE_LLDP, ether_types.ETH_TYPE_IPV6):
            return

        self.mac_to_port.setdefault(dp.id, {})[eth.src] = in_port

        # --- Política de aislamiento: H1 <-> H2 bloqueado hasta que la NBI lo autorice
        if self._is_isolated(pkt):
            self.logger.info("AISLAMIENTO: descartado %s -> %s en dpid=%s", *self._ip_pair(pkt), dp.id)
            return

        # --- Switch de aprendizaje L2 estándar
        out_port = self.mac_to_port[dp.id].get(eth.dst, ofp.OFPP_FLOOD)
        actions = [parser.OFPActionOutput(out_port)]
        if out_port != ofp.OFPP_FLOOD:
            match = parser.OFPMatch(in_port=in_port, eth_dst=eth.dst, eth_src=eth.src)
            self.add_flow(dp, LEARNING_PRIORITY, match, actions, cookie=LEARNING_COOKIE, idle_timeout=60)
        data = msg.data if msg.buffer_id == ofp.OFP_NO_BUFFER else None
        dp.send_msg(parser.OFPPacketOut(datapath=dp, buffer_id=msg.buffer_id,
                                        in_port=in_port, actions=actions, data=data))

    # ------------------------------------------------------------------ Servicio IP (NBI)
    def establish_connection(self, src_node, dst_node, bitrate_gbps, path=None):
        hosts, switches = TOPOLOGY["hosts"], TOPOLOGY["switches"]
        if src_node not in hosts or dst_node not in hosts:
            raise ValueError(f"Nodos desconocidos: {src_node}, {dst_node}")
        connection_id = f"{src_node}-{dst_node}"
        if connection_id in self.connections:   # idempotente: re-provisiona
            self.remove_connection(connection_id)
        path = path or default_path(src_node, dst_node)
        src_ip, dst_ip = hosts[src_node]["ip"], hosts[dst_node]["ip"]
        cookie = self._next_cookie
        self._next_cookie += 1

        flow_mods = 0
        for i in range(1, len(path) - 1):
            node = path[i]
            dp = self.datapaths.get(switches[node]["dpid"])
            if dp is None:
                raise RuntimeError(f"Switch {node} (dpid={switches[node]['dpid']}) no conectado")
            parser = dp.ofproto_parser
            fwd, rev = port_towards(node, path[i + 1]), port_towards(node, path[i - 1])
            for s_ip, d_ip, port in ((src_ip, dst_ip, fwd), (dst_ip, src_ip, rev)):
                actions = [parser.OFPActionOutput(port)]
                self.add_flow(dp, SERVICE_PRIORITY,
                              parser.OFPMatch(eth_type=ether_types.ETH_TYPE_IP, ipv4_src=s_ip, ipv4_dst=d_ip),
                              actions, cookie=cookie)
                self.add_flow(dp, SERVICE_PRIORITY,
                              parser.OFPMatch(eth_type=ether_types.ETH_TYPE_ARP, arp_spa=s_ip, arp_tpa=d_ip),
                              actions, cookie=cookie)
                flow_mods += 2

        self.isolated_pairs.discard(frozenset((src_ip, dst_ip)))
        self.connections[connection_id] = {
            "src_node": src_node, "dst_node": dst_node, "bitrate_gbps": bitrate_gbps,
            "path": path, "cookie": cookie, "flow_mods": flow_mods, "status": "UP",
        }
        self.logger.info("Conexión %s UP por %s (%d FLOW_MOD)", connection_id, path, flow_mods)
        return {"connection_id": connection_id, "status": "UP",
                "message": f"Connection {connection_id} established"}

    def remove_connection(self, connection_id):
        conn = self.connections.pop(connection_id, None)
        if conn is None:
            raise KeyError(connection_id)
        for dp in self.datapaths.values():
            self.delete_flows(dp, conn["cookie"])
            self.delete_flows(dp, LEARNING_COOKIE)
        hosts = TOPOLOGY["hosts"]
        self.isolated_pairs.add(frozenset((hosts[conn["src_node"]]["ip"], hosts[conn["dst_node"]]["ip"])))
        return {"connection_id": connection_id, "status": "DOWN",
                "message": f"Connection {connection_id} removed"}

    def status(self):
        return {
            "connections": {cid: {k: v for k, v in c.items() if k != "cookie"}
                            for cid, c in self.connections.items()},
            "isolated_pairs": [sorted(p) for p in self.isolated_pairs],
            "switches": sorted(self.datapaths),
        }


def _json(obj, status=200):
    return Response(status=status, content_type="application/json", charset="utf-8",
                    text=json.dumps(obj))


class IPServiceRestAPI(ControllerBase):
    """NBI REST del controlador (3 endpoints)."""

    def __init__(self, req, link, data, **config):
        super().__init__(req, link, data, **config)
        self.app = data[API_INSTANCE]

    @route("ipservice", "/service/ip/create", methods=["POST"])
    def create(self, req, **kwargs):
        try:
            body = json.loads(req.body.decode("utf-8") or "{}")
            result = self.app.establish_connection(
                body["src_node"], body["dst_node"], float(body.get("bitrate_gbps", 0)), body.get("path"))
        except KeyError as exc:
            return _json({"error": f"Falta el campo {exc}"}, 400)
        except (ValueError, RuntimeError) as exc:
            return _json({"error": str(exc)}, 409)
        return _json(result)

    @route("ipservice", "/service/ip/status", methods=["GET"])
    def get_status(self, req, **kwargs):
        return _json(self.app.status())

    @route("ipservice", "/service/ip/delete", methods=["DELETE"])
    def delete(self, req, **kwargs):
        connection_id = req.params.get("connection_id")
        if not connection_id and req.body:
            connection_id = json.loads(req.body.decode("utf-8")).get("connection_id")
        try:
            return _json(self.app.remove_connection(connection_id))
        except KeyError:
            return _json({"error": f"Conexión {connection_id} no existe"}, 404)


In [ ]:
%%writefile mininet_topo.py
#!/usr/bin/env python3
"""
mininet_topo.py
===============
Topología Mininet del laboratorio (OpenFlow 1.3 + controlador remoto Ryu):

    h1 (10.0.0.1) -- r1 -- r2 -- r3 -- h2 (10.0.0.2)

Modos de uso (requiere Linux + root):
    sudo python3 mininet_topo.py --mode cli     # CLI interactiva (mininet> h1 ping h2)
    sudo python3 mininet_topo.py --mode test    # ping h1->h2 y termina
    sudo python3 mininet_topo.py --mode serve   # mantiene la red y expone un canal HTTP
                                                # GET /ping?src=h1&dst=h2&count=3
"""
import argparse
import json
import os
import sys
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.parse import parse_qs, urlparse

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from mininet.cli import CLI                                   # noqa: E402
from mininet.log import info, setLogLevel                     # noqa: E402
from mininet.net import Mininet                               # noqa: E402
from mininet.node import OVSSwitch, RemoteController          # noqa: E402
from mininet.topo import Topo                                 # noqa: E402

from access_topology import TOPOLOGY                          # noqa: E402


class AccessAggregationTopo(Topo):
    """H1 (acceso) - R1 - R2 - R3 - H2 (borde), puertos fijos según access_topology.py."""

    def build(self):
        nodes = {}
        for hid, h in TOPOLOGY["hosts"].items():
            nodes[hid] = self.addHost(h["name"], ip=f"{h['ip']}/24", mac=h["mac"])
        for sid, s in TOPOLOGY["switches"].items():
            nodes[sid] = self.addSwitch(s["name"], dpid=f"{s['dpid']:016x}", protocols="OpenFlow13")
        for link in TOPOLOGY["links"]:
            if not link["in_mininet"]:
                continue  # el bypass óptico no se emula en Mininet
            opts = {}
            if link["a_port"] is not None:
                opts["port1"] = link["a_port"]
            if link["b_port"] is not None:
                opts["port2"] = link["b_port"]
            self.addLink(nodes[link["a"]], nodes[link["b"]], **opts)


def build_network(controller_ip, controller_port):
    net = Mininet(topo=AccessAggregationTopo(), switch=OVSSwitch, build=False,
                  controller=None, autoSetMacs=False)
    net.addController("ryu", controller=RemoteController, ip=controller_ip, port=controller_port)
    net.build()
    net.start()
    return net


def ping(net, src, dst, count=3):
    dst_ip = net.get(dst).IP()
    return net.get(src).cmd(f"ping -c {count} -W 1 {dst_ip}")


def serve(net, port):
    class Handler(BaseHTTPRequestHandler):
        def _send(self, code, obj):
            body = json.dumps(obj).encode()
            self.send_response(code)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(body)))
            self.end_headers()
            self.wfile.write(body)

        def do_GET(self):
            parsed = urlparse(self.path)
            qs = {k: v[0] for k, v in parse_qs(parsed.query).items()}
            if parsed.path == "/health":
                self._send(200, {"status": "ok", "hosts": [h.name for h in net.hosts]})
            elif parsed.path == "/ping":
                out = ping(net, qs.get("src", "h1"), qs.get("dst", "h2"), int(qs.get("count", 3)))
                self._send(200, {"output": out})
            else:
                self._send(404, {"error": "not found"})

        def log_message(self, *args):
            pass

    info(f"*** Canal de control Mininet en http://127.0.0.1:{port}\n")
    HTTPServer(("127.0.0.1", port), Handler).serve_forever()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--controller-ip", default="127.0.0.1")
    parser.add_argument("--controller-port", type=int, default=6653)
    parser.add_argument("--mode", choices=["cli", "test", "serve"], default="cli")
    parser.add_argument("--serve-port", type=int, default=9000)
    args = parser.parse_args()

    setLogLevel("info")
    net = build_network(args.controller_ip, args.controller_port)
    try:
        if args.mode == "test":
            info("*** Prueba de aislamiento inicial: h1 ping h2\n")
            print(ping(net, "h1", "h2"))
        elif args.mode == "serve":
            serve(net, args.serve_port)
        else:
            CLI(net)
    except KeyboardInterrupt:
        pass
    finally:
        net.stop()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile sdn_emulator.py
"""
sdn_emulator.py
===============
Simulador de respaldo *transparente* para ``SIMULATION_MODE = True``.

Reproduce, en el propio proceso de Jupyter y sin privilegios root:
  * Switches OVS con tablas de flujo OpenFlow 1.3 (match/prioridad/acciones/cookie).
  * Un plano de datos que reenvía ARP e ICMP salto a salto según esas tablas.
  * ``EmulatedRyuController``: MISMA lógica que ``ryu_controller.py`` (switch L2
    de aprendizaje + aislamiento H1<->H2 + servicio IP), registrando cada
    OFPT_FLOW_MOD / OFPT_PACKET_IN en la captura del workshop.
  * ``create_sdn_app``: MISMA NBI REST que Ryu (/service/ip/create|status|delete).
"""
import random
import threading
from typing import Any

from access_topology import TOPOLOGY, default_path, port_towards
import workshop_common as wc

BROADCAST = "ff:ff:ff:ff:ff:ff"
ETH_ARP, ETH_IP = 0x0806, 0x0800
LEARNING_COOKIE = 0x1000
SERVICE_COOKIE_BASE = 0xA000


# =============================================================================
# Plano de datos emulado (OVS)
# =============================================================================
class FlowEntry:
    def __init__(self, priority: int, match: dict, actions: list, cookie: int = 0):
        self.priority, self.match, self.actions, self.cookie = priority, match, actions, cookie
        self.n_packets = 0

    def matches(self, pkt: dict, in_port: int) -> bool:
        for key, value in self.match.items():
            actual = in_port if key == "in_port" else pkt.get(key)
            if actual != value:
                return False
        return True

    def __repr__(self) -> str:
        acts = ",".join(f"{a[0]}:{a[1]}" if len(a) > 1 else a[0] for a in self.actions) or "drop"
        match = ",".join(f"{k}={v:#06x}" if k == "eth_type" else f"{k}={v}" for k, v in self.match.items())
        return (f"cookie={self.cookie:#x}, priority={self.priority}, n_packets={self.n_packets}, "
                f"{match or '(any)'} actions={acts}")


class EmulatedSwitch:
    def __init__(self, sid: str, name: str, dpid: int):
        self.sid, self.name, self.dpid = sid, name, dpid
        self.ports: dict[int, tuple[str, int | None]] = {}
        self.flow_table: list[FlowEntry] = []

    def flow_mod(self, command: str, priority: int = 0, match: dict | None = None,
                 actions: list | None = None, cookie: int = 0) -> None:
        if command == "ADD":
            self.flow_table = [f for f in self.flow_table
                               if not (f.priority == priority and f.match == (match or {}))]
            self.flow_table.append(FlowEntry(priority, match or {}, actions or [], cookie))
            self.flow_table.sort(key=lambda f: -f.priority)
        elif command == "DELETE":
            self.flow_table = [f for f in self.flow_table if f.cookie != cookie]

    def lookup(self, pkt: dict, in_port: int) -> FlowEntry | None:
        for entry in self.flow_table:
            if entry.matches(pkt, in_port):
                entry.n_packets += 1
                return entry
        return None

    def dump_flows(self) -> str:
        """Equivalente a: ovs-ofctl -O OpenFlow13 dump-flows <switch>"""
        return "\n".join(f" {f}" for f in self.flow_table) or " (tabla vacía)"


class EmulatedNetwork:
    """Plano de datos: hosts + switches cableados según access_topology.py."""

    def __init__(self, topology: dict = TOPOLOGY):
        self.topology = topology
        self.hosts = topology["hosts"]
        self.switches = {sid: EmulatedSwitch(sid, s["name"], s["dpid"])
                         for sid, s in topology["switches"].items()}
        self.controller: EmulatedRyuController | None = None
        self._lock = threading.RLock()
        self.arp_cache: dict[str, dict[str, str]] = {h: {} for h in self.hosts}
        for link in topology["links"]:
            a, b = link["a"], link["b"]
            if a in self.switches and link["a_port"] is not None:
                self.switches[a].ports[link["a_port"]] = (b, link["b_port"])
            if b in self.switches and link["b_port"] is not None:
                self.switches[b].ports[link["b_port"]] = (a, link["a_port"])
        self._link_state = {frozenset((l["a"], l["b"])): l for l in topology["links"]}

    def attach_controller(self, controller: "EmulatedRyuController") -> None:
        self.controller = controller
        for sw in self.switches.values():
            controller.switch_connected(sw)

    def _port_up(self, sw: EmulatedSwitch, port: int) -> bool:
        peer = sw.ports[port][0]
        return self._link_state[frozenset((sw.sid, peer))]["state"] == "up"

    # --------------------------------------------------------------- reenvío
    def _process(self, sw: EmulatedSwitch, in_port: int, pkt: dict, ctx: dict) -> None:
        ctx["hops"] += 1
        if ctx["hops"] > 64:
            return
        entry = sw.lookup(pkt, in_port)
        if entry is None:
            return
        for action in entry.actions:
            if action[0] == "CONTROLLER":
                ctx["packet_ins"] += 1
                for out_port in self.controller.packet_in(sw, in_port, pkt):
                    self._emit(sw, out_port, in_port, pkt, ctx)
            elif action[0] == "OUTPUT":
                self._emit(sw, action[1], in_port, pkt, ctx)

    def _emit(self, sw: EmulatedSwitch, out_port: Any, in_port: int, pkt: dict, ctx: dict) -> None:
        ports = ([p for p in sw.ports if p != in_port and self._port_up(sw, p)]
                 if out_port == "FLOOD" else [out_port])
        for port in ports:
            if port not in sw.ports or (sw.sid, port) in ctx["visited"]:
                continue
            ctx["visited"].add((sw.sid, port))
            peer, peer_port = sw.ports[port]
            if peer in self.hosts:
                if pkt["eth_dst"] in (BROADCAST, self.hosts[peer]["mac"]):
                    ctx["delivered"].append(peer)
            else:
                self._process(self.switches[peer], peer_port, pkt, ctx)

    def send(self, host: str, pkt: dict) -> dict:
        h = self.hosts[host]
        ctx = {"hops": 0, "packet_ins": 0, "delivered": [], "visited": set()}
        self._process(self.switches[h["switch"]], h["port"], pkt, ctx)
        return ctx

    # --------------------------------------------------------------- ping
    def _arp_resolve(self, src: str, dst: str) -> tuple[bool, dict]:
        s, d = self.hosts[src], self.hosts[dst]
        req = {"eth_src": s["mac"], "eth_dst": BROADCAST, "eth_type": ETH_ARP,
               "arp_spa": s["ip"], "arp_tpa": d["ip"], "arp_op": 1}
        ctx = self.send(src, req)
        if dst not in ctx["delivered"]:
            return False, ctx
        rep = {"eth_src": d["mac"], "eth_dst": s["mac"], "eth_type": ETH_ARP,
               "arp_spa": d["ip"], "arp_tpa": s["ip"], "arp_op": 2}
        ctx2 = self.send(dst, rep)
        ctx["packet_ins"] += ctx2["packet_ins"]
        if src not in ctx2["delivered"]:
            return False, ctx
        self.arp_cache[src][d["ip"]] = d["mac"]
        self.arp_cache[dst][s["ip"]] = s["mac"]
        return True, ctx

    def ping(self, src: str = "H1", dst: str = "H2", count: int = 3) -> str:
        """Emula ``mininet> h1 ping h2`` con salida idéntica en formato a iputils-ping."""
        with self._lock:
            s, d = self.hosts[src], self.hosts[dst]
            lines = [f"PING {d['ip']} ({d['ip']}) 56(84) bytes of data."]
            received, errors, rtts = 0, 0, []
            for seq in range(1, count + 1):
                extra_ms = 0.0
                if d["ip"] not in self.arp_cache[src]:
                    ok, ctx = self._arp_resolve(src, dst)
                    if not ok:
                        errors += 1
                        lines.append(f"From {s['ip']} icmp_seq={seq} Destination Host Unreachable")
                        continue
                    extra_ms += random.uniform(8.0, 11.0) + 3.0 * ctx["packet_ins"]
                echo = {"eth_src": s["mac"], "eth_dst": d["mac"], "eth_type": ETH_IP,
                        "ipv4_src": s["ip"], "ipv4_dst": d["ip"], "icmp_type": 8}
                ctx = self.send(src, echo)
                if dst not in ctx["delivered"]:
                    self.arp_cache[src].pop(d["ip"], None)
                    continue
                reply = {"eth_src": d["mac"], "eth_dst": s["mac"], "eth_type": ETH_IP,
                         "ipv4_src": d["ip"], "ipv4_dst": s["ip"], "icmp_type": 0}
                ctx2 = self.send(dst, reply)
                if src not in ctx2["delivered"]:
                    continue
                base = 0.25 if seq == 2 else 0.07
                rtt = extra_ms + base + 0.004 * (ctx["hops"] + ctx2["hops"]) + random.uniform(0, 0.04)
                rtt += 3.0 * (ctx["packet_ins"] + ctx2["packet_ins"])
                rtts.append(rtt)
                received += 1
                rtt_txt = f"{rtt:.1f}" if rtt >= 10 else (f"{rtt:.2f}" if rtt >= 1 else f"{rtt:.3f}")
                lines.append(f"64 bytes from {d['ip']}: icmp_seq={seq} ttl=64 time={rtt_txt} ms")
            loss = 100 * (count - received) // count
            lines += ["", f"--- {d['ip']} ping statistics ---"]
            err_txt = f", +{errors} errors" if errors else ""
            lines.append(f"{count} packets transmitted, {received} received{err_txt}, "
                         f"{loss}% packet loss, time {1000 * (count - 1) + 3}ms")
            if rtts:
                avg = sum(rtts) / len(rtts)
                lines.append(f"rtt min/avg/max/mdev = {min(rtts):.3f}/{avg:.3f}/{max(rtts):.3f}/"
                             f"{(max(rtts) - min(rtts)) / 2:.3f} ms")
            # caché ARP con vida corta: cada ping vuelve a validar la conectividad
            self.arp_cache = {h: {} for h in self.hosts}
            return "\n".join(lines)

    def dump_flows(self, sid: str) -> str:
        sw = self.switches[sid]
        return f"$ ovs-ofctl -O OpenFlow13 dump-flows {sw.name}\n{sw.dump_flows()}"


# =============================================================================
# Controlador emulado: misma lógica que ryu_controller.py
# =============================================================================
class EmulatedRyuController:
    def __init__(self, network: EmulatedNetwork, label: str = "SDN"):
        self.net = network
        self.label = label
        self.mac_to_port: dict[int, dict[str, int]] = {}
        self.connections: dict[str, dict] = {}
        self._next_cookie = SERVICE_COOKIE_BASE
        h = network.hosts
        self.isolated_pairs = {frozenset((h["H1"]["ip"], h["H2"]["ip"]))}
        network.attach_controller(self)

    def _flow_mod(self, sw: EmulatedSwitch, command: str, priority: int = 0, match: dict | None = None,
                  actions: list | None = None, cookie: int = 0, log: bool = True) -> None:
        sw.flow_mod(command, priority, match, actions, cookie)
        if log:
            if command == "ADD":
                desc = ", ".join(f"{k}={v:#06x}" if k == "eth_type" else f"{k}={v}"
                                 for k, v in (match or {}).items()) or "table-miss"
                acts = ",".join(f"{a[0]}:{a[1]}" if len(a) > 1 else a[0] for a in (actions or []))
                info = f"Type: OFPT_FLOW_MOD (ADD prio={priority} {desc} -> {acts})"
            else:
                info = f"Type: OFPT_FLOW_MOD (DELETE cookie={cookie:#x})"
            wc.CAPTURE.record(self.label, f"OVS {sw.name}", "OpenFlow", info, length=80 + 16 * len(match or {}))

    def switch_connected(self, sw: EmulatedSwitch) -> None:
        self._flow_mod(sw, "ADD", 0, {}, [("CONTROLLER",)], log=False)

    @staticmethod
    def _ip_pair(pkt: dict) -> tuple[str, str] | None:
        if pkt.get("eth_type") == ETH_ARP:
            return pkt["arp_spa"], pkt["arp_tpa"]
        if pkt.get("eth_type") == ETH_IP:
            return pkt["ipv4_src"], pkt["ipv4_dst"]
        return None

    def packet_in(self, sw: EmulatedSwitch, in_port: int, pkt: dict) -> list:
        wc.CAPTURE.record(f"OVS {sw.name}", self.label, "OpenFlow",
                          f"Type: OFPT_PACKET_IN (in_port={in_port}, eth_type={pkt['eth_type']:#06x})")
        self.mac_to_port.setdefault(sw.dpid, {})[pkt["eth_src"]] = in_port
        pair = self._ip_pair(pkt)
        if pair and frozenset(pair) in self.isolated_pairs:
            return []                                   # AISLAMIENTO: descartado
        out = self.mac_to_port[sw.dpid].get(pkt["eth_dst"], "FLOOD")
        if out != "FLOOD":
            self._flow_mod(sw, "ADD", 1, {"in_port": in_port, "eth_dst": pkt["eth_dst"],
                                          "eth_src": pkt["eth_src"]}, [("OUTPUT", out)], LEARNING_COOKIE)
        return [out]

    # --------------------------------------------------------------- NBI
    def establish_connection(self, src_node: str, dst_node: str, bitrate_gbps: float,
                             path: list[str] | None = None) -> dict:
        hosts = self.net.hosts
        if src_node not in hosts or dst_node not in hosts:
            raise ValueError(f"Nodos desconocidos: {src_node}, {dst_node}")
        connection_id = f"{src_node}-{dst_node}"
        if connection_id in self.connections:
            self.remove_connection(connection_id)
        path = path or default_path(src_node, dst_node, self.net.topology)
        src_ip, dst_ip = hosts[src_node]["ip"], hosts[dst_node]["ip"]
        cookie = self._next_cookie
        self._next_cookie += 1
        flow_mods = 0
        for i in range(1, len(path) - 1):
            sw = self.net.switches[path[i]]
            fwd = port_towards(path[i], path[i + 1], self.net.topology)
            rev = port_towards(path[i], path[i - 1], self.net.topology)
            for s_ip, d_ip, port in ((src_ip, dst_ip, fwd), (dst_ip, src_ip, rev)):
                self._flow_mod(sw, "ADD", 100, {"eth_type": ETH_IP, "ipv4_src": s_ip, "ipv4_dst": d_ip},
                               [("OUTPUT", port)], cookie)
                self._flow_mod(sw, "ADD", 100, {"eth_type": ETH_ARP, "arp_spa": s_ip, "arp_tpa": d_ip},
                               [("OUTPUT", port)], cookie)
                flow_mods += 2
        self.isolated_pairs.discard(frozenset((src_ip, dst_ip)))
        self.connections[connection_id] = {"src_node": src_node, "dst_node": dst_node,
                                           "bitrate_gbps": bitrate_gbps, "path": path,
                                           "cookie": cookie, "flow_mods": flow_mods, "status": "UP"}
        return {"connection_id": connection_id, "status": "UP",
                "message": f"Connection {connection_id} established"}

    def remove_connection(self, connection_id: str) -> dict:
        conn = self.connections.pop(connection_id, None)
        if conn is None:
            raise KeyError(connection_id)
        for sw in self.net.switches.values():
            self._flow_mod(sw, "DELETE", cookie=conn["cookie"])
            sw.flow_mod("DELETE", cookie=LEARNING_COOKIE)
        hosts = self.net.hosts
        self.isolated_pairs.add(frozenset((hosts[conn["src_node"]]["ip"], hosts[conn["dst_node"]]["ip"])))
        return {"connection_id": connection_id, "status": "DOWN",
                "message": f"Connection {connection_id} removed"}

    def status(self) -> dict:
        return {
            "connections": {cid: {k: v for k, v in c.items() if k != "cookie"}
                            for cid, c in self.connections.items()},
            "isolated_pairs": [sorted(p) for p in self.isolated_pairs],
            "switches": sorted(sw.dpid for sw in self.net.switches.values()),
        }


def create_sdn_app(controller: EmulatedRyuController):
    """NBI REST idéntica a la de ryu_controller.py, servida con FastAPI."""
    from fastapi import FastAPI, HTTPException
    from pydantic import BaseModel

    class CreateRequest(BaseModel):
        src_node: str
        dst_node: str
        bitrate_gbps: float = 0.0
        path: list[str] | None = None

    app = FastAPI(title="Ryu SDN NBI (emulada)")

    @app.post("/service/ip/create")
    def create(req: CreateRequest):
        try:
            with controller.net._lock:
                return controller.establish_connection(req.src_node, req.dst_node, req.bitrate_gbps, req.path)
        except (ValueError, RuntimeError) as exc:
            raise HTTPException(status_code=409, detail=str(exc))

    @app.get("/service/ip/status")
    def status():
        return controller.status()

    @app.delete("/service/ip/delete")
    def delete(connection_id: str):
        try:
            with controller.net._lock:
                return controller.remove_connection(connection_id)
        except KeyError:
            raise HTTPException(status_code=404, detail=f"Conexión {connection_id} no existe")

    return app


In [ ]:
%%writefile tfs_adapter.py
"""
tfs_adapter.py
==============
Adaptador de la herramienta MCP ``establish_ip_connection`` hacia ETSI
TeraFlowSDN (TFS), el controlador SDN *cloud-native* de ETSI.

Mapeo conceptual Ryu -> TFS:
    POST /service/ip/create (Ryu)  ==>  NBI "TFS API":
        POST /tfs-api/context/{context_uuid}/services
    que el microservicio NBI traduce a la RPC gRPC ``ServiceService.CreateService``
    + ``UpdateService``. ``Service`` invoca a ``PathComp`` para la ruta y a
    ``Device`` (drivers OpenFlow/OpenConfig/TAPI...) para configurar los equipos.

El descriptor sigue el formato protobuf-JSON de los *descriptors* de TFS
(context.proto). Verifique los campos exactos contra el Swagger de su despliegue.
Con ``dry_run=True`` NO se contacta a ningún TFS real: se devuelve el descriptor.
"""
from __future__ import annotations

from access_topology import TOPOLOGY, port_towards
import workshop_common as wc


class TeraFlowSDNAdapter:
    def __init__(self, base_url: str = "http://127.0.0.1:80", context_uuid: str = "admin",
                 dry_run: bool = True):
        self.base_url = base_url.rstrip("/")
        self.context_uuid = context_uuid
        self.dry_run = dry_run

    def _endpoint(self, device: str, port: int) -> dict:
        return {"device_id": {"device_uuid": {"uuid": device}},
                "endpoint_uuid": {"uuid": str(port)}}

    def build_service_descriptor(self, src_node: str, dst_node: str, bitrate_gbps: float,
                                 path: list[str] | None = None,
                                 service_type: str = "SERVICETYPE_L3NM") -> dict:
        """Construye el descriptor de un servicio L3NM (o L2NM / TAPI_CONNECTIVITY_SERVICE)."""
        hosts = TOPOLOGY["hosts"]
        src_sw, dst_sw = hosts[src_node]["switch"], hosts[dst_node]["switch"]
        service_uuid = f"{src_node}-{dst_node}"
        descriptor = {
            "service_id": {"context_id": {"context_uuid": {"uuid": self.context_uuid}},
                           "service_uuid": {"uuid": service_uuid}},
            "name": service_uuid,
            "service_type": service_type,
            "service_status": {"service_status": "SERVICESTATUS_PLANNED"},
            "service_endpoint_ids": [
                self._endpoint(src_sw, port_towards(src_sw, src_node)),
                self._endpoint(dst_sw, port_towards(dst_sw, dst_node)),
            ],
            "service_constraints": [
                {"custom": {"constraint_type": "bandwidth[gbps]", "constraint_value": str(float(bitrate_gbps))}},
            ],
            "service_config": {"config_rules": [
                {"action": "CONFIGACTION_SET", "custom": {
                    "resource_key": "/settings",
                    "resource_value": {"src_ip": hosts[src_node]["ip"], "dst_ip": hosts[dst_node]["ip"]}}},
            ]},
        }
        if path:
            # Ruta sugerida por el PCE externo: TFS la recibe como restricción (en
            # producción la calcularía su propio microservicio PathComp).
            descriptor["service_constraints"].append(
                {"custom": {"constraint_type": "explicit_route", "constraint_value": ",".join(path)}})
        return {"services": [descriptor]}

    def create_ip_service(self, src_node: str, dst_node: str, bitrate_gbps: float,
                          path: list[str] | None = None) -> dict:
        body = self.build_service_descriptor(src_node, dst_node, bitrate_gbps, path)
        endpoint = f"{self.base_url}/tfs-api/context/{self.context_uuid}/services"
        service_uuid = f"{src_node}-{dst_node}"
        if self.dry_run:
            wc.CAPTURE.record("Agent", "TFS (dry)", "HTTP",
                              f"POST /tfs-api/context/{self.context_uuid}/services HTTP/1.1  (dry-run)", body)
        else:
            wc.http_call("Agent", "TFS", "POST", endpoint, body)
        return {"connection_id": service_uuid, "status": "UP",
                "message": f"Connection {service_uuid} established"}

    def delete_ip_service(self, connection_id: str) -> dict:
        endpoint = f"{self.base_url}/tfs-api/context/{self.context_uuid}/service/{connection_id}"
        if self.dry_run:
            wc.CAPTURE.record("Agent", "TFS (dry)", "HTTP",
                              f"DELETE /tfs-api/context/{self.context_uuid}/service/{connection_id} (dry-run)")
        else:
            wc.http_call("Agent", "TFS", "DELETE", endpoint)
        return {"connection_id": connection_id, "status": "DOWN",
                "message": f"Connection {connection_id} removed"}


In [ ]:
# ⚙️ Arranque del plano de datos + controlador SDN (real o simulado)
import subprocess
from access_topology import TOPOLOGY

if SIMULATION_MODE:
    sdn_emulator = load_module("sdn_emulator")
    net = sdn_emulator.EmulatedNetwork()                     # OVS r1, r2, r3 + hosts h1, h2
    sdn_controller = sdn_emulator.EmulatedRyuController(net)  # misma lógica que ryu_controller.py
    SDN_URL = wc.start_server("sdn", sdn_emulator.create_sdn_app(sdn_controller), "sdn")
else:
    host, ports = wc.CONFIG["HOST"], wc.CONFIG["PORTS"]
    if not wc.port_in_use(host, ports["sdn"]):
        ryu_proc = subprocess.Popen(["ryu-manager", "ryu_controller.py", "--ofp-tcp-listen-port", "6653",
                                     "--wsapi-port", str(ports["sdn"])], stdout=open("ryu.log", "w"),
                                    stderr=subprocess.STDOUT)
        assert wc.wait_for_port(host, ports["sdn"], 30), "Ryu no levantó: revise ryu.log"
    if not wc.port_in_use(host, ports["mininet_ctl"]):
        mn_proc = subprocess.Popen(["sudo", "-n", "python3", "mininet_topo.py", "--mode", "serve",
                                    "--serve-port", str(ports["mininet_ctl"])], stdout=open("mininet.log", "w"),
                                   stderr=subprocess.STDOUT)
        assert wc.wait_for_port(host, ports["mininet_ctl"], 90), "Mininet no levantó: revise mininet.log"
    time.sleep(3)
    SDN_URL = wc.url("sdn")


def mininet_ping(src="H1", dst="H2", count=3):
    # Equivale a 'mininet> h1 ping h2' (real o emulado)
    print(f"mininet> {src.lower()} ping {dst.lower()}")
    if SIMULATION_MODE:
        print(net.ping(src, dst, count))
    else:
        r = requests.get(f"{wc.url('mininet_ctl')}/ping", params={"src": src.lower(), "dst": dst.lower(),
                         "count": count}, timeout=60)
        print(r.json()["output"])


def dump_flows(switch="R2"):
    # Equivale a 'ovs-ofctl -O OpenFlow13 dump-flows <switch>'
    if SIMULATION_MODE:
        print(net.dump_flows(switch))
    else:
        print(subprocess.run(["sudo", "-n", "ovs-ofctl", "-O", "OpenFlow13", "dump-flows", switch.lower()],
                             capture_output=True, text=True).stdout)


def connection_path():
    conn = requests.get(f"{SDN_URL}/service/ip/status", timeout=10).json()["connections"].get("H1-H2")
    return conn["path"] if conn else None

print(f"✅ Plano de datos {'emulado' if SIMULATION_MODE else 'Mininet'} + controlador Ryu (NBI REST) en {SDN_URL}")

In [ ]:
# ▶️ ¿Se alcanzan H1 (acceso) y H2 (borde)?
viz.plot_network(TOPOLOGY, "Estación 1 · Red de acceso/agregación antes de cualquier intención", isolated=True)
mininet_ping("H1", "H2")

In [ ]:
# ▶️ ¿Qué reglas tiene el switch R1? (solo la regla 'table-miss' -> CONTROLLER)
dump_flows("R1")
print("\nNBI del controlador · GET /service/ip/status")
print(wc.pretty(requests.get(f"{SDN_URL}/service/ip/status", timeout=10).json()))

👀 **Observe:** el ping falla con `Destination Host Unreachable` y R1 solo tiene la regla de prioridad 0 que envía todo al controlador (`n_packets` creció con los ARP de H1 que Ryu descartó).

💬 **Pregunta rápida:** ¿por qué basta con descartar el **ARP** para que el ping falle?

---
## 🟦🟦🟦⬜⬜⬜ Estación 2 · Herramientas deterministas: PCE + MCP · ⏱️ 8 min
🎯 **Objetivo:** conocer las herramientas que **sí** tocan la red y cómo se exponen a los agentes.

📖 **Concepto en 1 minuto** *(Diapositiva 9)*
* **Inventario (NetworkX) + PCE:** la red es un grafo; el PCE calcula la ruta de menos saltos sobre los enlaces `up`. **Misma entrada ⇒ misma salida**: el LLM nunca calcula rutas.
* **MCP (Model Context Protocol):** el agente pregunta `tools/list` y recibe, por cada herramienta, **nombre + descripción + JSON Schema**; luego invoca `tools/call`. Con **FastMCP** basta decorar una función:

```python
@mcp.tool
def establish_ip_connection(src_node: str, dst_node: str, bitrate_gbps: float) -> dict:
    """Provision and bring up a new IP connection on the SDN-controlled network..."""
    return http_call("POST", SDN + "/service/ip/create", ...)   # NBI de Ryu / TFS
```

> ⚙️ **Código de soporte (celdas plegadas).** Las siguientes celdas escriben/arrancan los componentes de esta estación: `network_inventory.py (inventario + PCE)` · `mcp_server.py (servidor FastMCP con 4 herramientas)`.
> **Solo ejecútelas** (Shift+Enter); no es necesario leerlas en clase. Puede desplegarlas para ver el detalle completo.

In [ ]:
%%writefile network_inventory.py
"""
network_inventory.py
====================
Herramientas DETERMINISTAS de red usadas por el agente vía MCP:

  * ``NetworkInventory``: inventario/estado de red sobre un grafo NetworkX
    (nodos H1, R1, R2, R3, H2; capacidad, uso, capa y estado de cada enlace).
  * ``PathComputationElement`` (PCE): cálculo de ruta más corta (en saltos)
    sobre los enlaces operativos. Misma entrada => misma salida, siempre.
  * ``create_pce_app``: expone el PCE como servicio REST (POST /compute_route).
"""
import copy

import networkx as nx

from access_topology import TOPOLOGY


class RouteNotFoundError(Exception):
    pass


class NetworkInventory:
    def __init__(self, topology: dict = TOPOLOGY):
        self.topology = copy.deepcopy(topology)
        self.graph = nx.Graph()
        for hid, h in self.topology["hosts"].items():
            self.graph.add_node(hid, kind="host", **h)
        for sid, s in self.topology["switches"].items():
            self.graph.add_node(sid, kind="switch", **s)
        for link in self.topology["links"]:
            attrs = {k: v for k, v in link.items() if k not in ("a", "b")}
            self.graph.add_edge(link["a"], link["b"], **attrs)

    def residual_gbps(self, u: str, v: str) -> float:
        e = self.graph.edges[u, v]
        return e["capacity_gbps"] - e["used_gbps"]

    def set_link_state(self, u: str, v: str, state: str) -> None:
        self.graph.edges[u, v]["state"] = state

    def operational_graph(self) -> nx.Graph:
        """Subgrafo con los enlaces en estado 'up'."""
        edges = [(u, v) for u, v, d in self.graph.edges(data=True) if d["state"] == "up"]
        return self.graph.edge_subgraph(edges).copy()

    def reserve(self, path: list[str], bitrate_gbps: float) -> None:
        for u, v in zip(path, path[1:]):
            self.graph.edges[u, v]["used_gbps"] += bitrate_gbps

    def links_table(self) -> list[dict]:
        rows = []
        for u, v, d in self.graph.edges(data=True):
            rows.append({"link": f"{u}-{v}", "layer": d["layer"], "state": d["state"],
                         "capacity_gbps": d["capacity_gbps"], "used_gbps": d["used_gbps"],
                         "residual_gbps": d["capacity_gbps"] - d["used_gbps"]})
        return rows


class PathComputationElement:
    """PCE determinista: ruta de mínimo número de saltos sobre la red operativa."""

    def __init__(self, inventory: NetworkInventory):
        self.inventory = inventory

    def compute_route(self, src: str, dst: str) -> dict:
        graph = self.inventory.operational_graph()
        for node in (src, dst):
            if node not in self.inventory.graph:
                raise RouteNotFoundError(f"Nodo desconocido: {node}")
        try:
            path = nx.shortest_path(graph, src, dst)
        except (nx.NetworkXNoPath, nx.NodeNotFound) as exc:
            raise RouteNotFoundError(f"No existe ruta {src}->{dst}") from exc
        return {"src": src, "dst": dst, "path": path, "hops": len(path) - 1}


def create_pce_app(pce: PathComputationElement):
    from fastapi import FastAPI, HTTPException
    from pydantic import BaseModel

    class RouteRequest(BaseModel):
        src_node: str
        dst_node: str

    app = FastAPI(title="Path Computation Element (PCE)")

    @app.post("/compute_route")
    def compute_route(req: RouteRequest):
        try:
            return pce.compute_route(req.src_node, req.dst_node)
        except RouteNotFoundError as exc:
            raise HTTPException(status_code=404, detail=str(exc))

    @app.get("/topology")
    def topology():
        return {"nodes": list(pce.inventory.graph.nodes), "links": pce.inventory.links_table()}

    return app


In [ ]:
%%writefile mcp_server.py
"""
mcp_server.py
=============
Servidor MCP (Model Context Protocol) construido con FastMCP. Expone como
*Tools* las capacidades DETERMINISTAS de la red:

    compute_route(src_node, dst_node)                      -> PCE   (POST /compute_route)
    establish_ip_connection(src_node, dst_node, bitrate)   -> NBI SDN (POST /service/ip/create)
    remove_ip_connection(src_node, dst_node)               -> NBI SDN (DELETE /service/ip/delete)
    get_connection_status()                                -> NBI SDN (GET /service/ip/status)

El LLM NUNCA calcula rutas ni programa flujos: solo elige herramientas y
argumentos. La ejecución es código determinista y auditable.

Uso standalone (transporte HTTP "streamable"):
    python mcp_server.py      # -> http://127.0.0.1:8005/mcp
"""
from __future__ import annotations

import inspect
import json
from typing import Annotated, Any, Callable

from pydantic import Field

import workshop_common as wc

try:
    from fastmcp import Client, FastMCP
    FASTMCP_AVAILABLE = True
except ImportError:  # pragma: no cover - respaldo mínimo compatible
    FASTMCP_AVAILABLE = False

    class FastMCP:  # type: ignore[no-redef]
        """Implementación mínima (solo para entornos sin fastmcp)."""

        def __init__(self, name: str, instructions: str = ""):
            self.name, self.instructions = name, instructions
            self._tools: dict[str, Callable] = {}

        def tool(self, fn: Callable | None = None, *, name: str | None = None, description: str | None = None):
            def register(f: Callable) -> Callable:
                f.__mcp_description__ = description or inspect.getdoc(f) or ""
                self._tools[name or f.__name__] = f
                return f
            return register(fn) if fn is not None else register

    Client = None  # type: ignore[assignment]


def _schema_for(fn: Callable) -> dict:
    from pydantic import create_model
    fields = {}
    for pname, param in inspect.signature(fn).parameters.items():
        default = ... if param.default is inspect.Parameter.empty else param.default
        fields[pname] = (param.annotation, default)
    schema = create_model(f"{fn.__name__}_args", **fields).model_json_schema()
    schema.pop("title", None)
    for prop in schema.get("properties", {}).values():
        prop.pop("title", None)
    return schema


mcp = FastMCP(
    name="TelecomNetworkMCP",
    instructions="Herramientas deterministas de red: PCE e interfaz Norte (NBI) del controlador SDN.",
)

# Caché de rutas calculadas por el PCE: establish_ip_connection reutiliza la ruta
_ROUTE_CACHE: dict[tuple[str, str], list[str]] = {}

NodeName = Annotated[str, Field(description="Network node identifier, e.g. 'H1', 'R2', 'H2'")]


def _sdn_backend():
    if wc.CONFIG.get("SDN_BACKEND") == "tfs":
        from tfs_adapter import TeraFlowSDNAdapter
        return TeraFlowSDNAdapter(wc.CONFIG["TFS_URL"], wc.CONFIG["TFS_CONTEXT"], wc.CONFIG["TFS_DRY_RUN"])
    return None


@mcp.tool
def compute_route(src_node: NodeName, dst_node: NodeName) -> dict:
    """Compute the shortest route between source and destination nodes for a candidate
    connection, using the deterministic Path Computation Element (PCE)."""
    result = wc.http_call("Agent", "PCE", "POST", f"{wc.url('pce')}/compute_route",
                          {"src_node": src_node, "dst_node": dst_node})
    _ROUTE_CACHE[(src_node, dst_node)] = result["path"]
    return result


@mcp.tool
def establish_ip_connection(
    src_node: NodeName,
    dst_node: NodeName,
    bitrate_gbps: Annotated[float, Field(description="Requested bandwidth in Gb/s", gt=0)],
) -> dict:
    """Provision and bring up a new IP connection on the SDN-controlled network through
    the SDN controller Northbound Interface (installs OpenFlow rules along the route)."""
    path = _ROUTE_CACHE.get((src_node, dst_node))
    tfs = _sdn_backend()
    if tfs is not None:
        return tfs.create_ip_service(src_node, dst_node, bitrate_gbps, path)
    return wc.http_call("Agent", "SDN", "POST", f"{wc.url('sdn')}/service/ip/create",
                        {"src_node": src_node, "dst_node": dst_node,
                         "bitrate_gbps": bitrate_gbps, "path": path})


@mcp.tool
def remove_ip_connection(src_node: NodeName, dst_node: NodeName) -> dict:
    """Tear down (remove/delete) an existing IP connection on the SDN-controlled network
    and restore the default isolation policy."""
    connection_id = f"{src_node}-{dst_node}"
    tfs = _sdn_backend()
    if tfs is not None:
        return tfs.delete_ip_service(connection_id)
    return wc.http_call("Agent", "SDN", "DELETE",
                        f"{wc.url('sdn')}/service/ip/delete?connection_id={connection_id}")


@mcp.tool
def get_connection_status() -> dict:
    """Retrieve the status of the IP connections and isolation policies from the SDN controller."""
    return wc.http_call("Agent", "SDN", "GET", f"{wc.url('sdn')}/service/ip/status")


# =============================================================================
# Cliente MCP utilizado por los agentes (sesión + descubrimiento + invocación)
# =============================================================================
class MCPSession:
    """Sesión MCP del agente. Usa transporte en memoria (por defecto) o HTTP."""

    def __init__(self, server: Any = None, caller: str = "Agent", server_url: str | None = None):
        self.server = server if server is not None else mcp
        self.caller = caller
        self.server_url = server_url
        self._client = None

    async def __aenter__(self) -> "MCPSession":
        wc.CAPTURE.record(self.caller, "MCP Server", "MCP", "JSON-RPC initialize")
        if FASTMCP_AVAILABLE:
            self._client = Client(self.server_url or self.server)
            await self._client.__aenter__()
        return self

    async def __aexit__(self, *exc: Any) -> None:
        if self._client is not None:
            await self._client.__aexit__(*exc)
            self._client = None

    async def list_tools(self) -> list[dict]:
        wc.CAPTURE.record(self.caller, "MCP Server", "MCP", "JSON-RPC tools/list")
        if FASTMCP_AVAILABLE:
            tools = []
            for t in await self._client.list_tools():
                schema = getattr(t, "input_schema", None)
                if schema is None:
                    schema = t.inputSchema
                tools.append({"name": t.name, "description": t.description or "", "inputSchema": schema})
        else:
            tools = [{"name": n, "description": f.__mcp_description__, "inputSchema": _schema_for(f)}
                     for n, f in self.server._tools.items()]
        wc.CAPTURE.record("MCP Server", self.caller, "MCP",
                          f"JSON-RPC result: {len(tools)} tools", [t["name"] for t in tools])
        return tools

    async def call_tool(self, name: str, arguments: dict) -> Any:
        wc.CAPTURE.record(self.caller, "MCP Server", "MCP", f"JSON-RPC tools/call name={name}", arguments)
        if FASTMCP_AVAILABLE:
            res = await self._client.call_tool(name, arguments)
            data = res.structured_content
            if isinstance(data, dict) and set(data) == {"result"}:
                data = data["result"]
            if data is None:
                data = getattr(res, "data", None)
            if data is None and res.content:
                data = json.loads(res.content[0].text)
        else:
            data = self.server._tools[name](**arguments)
        wc.CAPTURE.record("MCP Server", self.caller, "MCP", f"JSON-RPC result ({name})", data)
        return data


if __name__ == "__main__":
    mcp.run(transport="http", host=wc.CONFIG["HOST"], port=wc.CONFIG["PORTS"]["mcp_http"])


In [ ]:
# ▶️ Inventario + PCE (publicado como servicio REST: POST /compute_route)
network_inventory = load_module("network_inventory")
inventory = network_inventory.NetworkInventory()
pce = network_inventory.PathComputationElement(inventory)
PCE_URL = wc.start_server("pce", network_inventory.create_pce_app(pce), "pce")

route = requests.post(f"{PCE_URL}/compute_route", json={"src_node": "H1", "dst_node": "H2"}, timeout=10).json()
print("POST /compute_route ->", route)
viz.plot_network(inventory, "Estación 2 · Ruta calculada por el PCE (aún NO provisionada)",
                 computed_path=route["path"], isolated=True)

In [ ]:
# ▶️ ¿Qué ve un agente cuando abre una sesión MCP? (tools/list)
mcp_server = load_module("mcp_server")
from mcp_server import mcp, MCPSession

async with MCPSession(caller="Notebook") as session:
    tools = await session.list_tools()

print(f"{'Herramienta MCP':<26} {'Argumentos (tipo)':<58} Descripción")
print("-" * 130)
for t in tools:
    args = ", ".join(f"{k}: {v.get('type')}" for k, v in t["inputSchema"].get("properties", {}).items()) or "—"
    print(f"{t['name']:<26} {args:<58} {' '.join(t['description'].split())[:45]}…")
print("\nJSON Schema de establish_ip_connection:")
print(json.dumps(next(t for t in tools if t["name"] == "establish_ip_connection")["inputSchema"], indent=1))

👀 **Observe:** la ruta `H1 → R1 → R2 → R3 → H2` (4 saltos) es la del paper; el bypass óptico R1–R3 está en *standby* y no se usa. El JSON Schema exige `bitrate_gbps > 0`: un argumento inválido del LLM se rechaza **antes** de tocar la red.

💬 **Pregunta rápida:** si mañana cambiamos Ryu por TeraFlowSDN, ¿qué parte de esta estación cambiaría y cuál no?

---
## 🟦🟦🟦🟦⬜⬜ Estación 3 · El "cerebro": ChromaDB + LLM local · ⏱️ 8 min
🎯 **Objetivo:** ver cómo una frase en lenguaje natural se convierte en un **plan estructurado** sin que el LLM invente procedimientos.

📖 **Concepto en 1 minuto** *(Diapositiva 10)*
* **Zero-shot:** el prompt no lleva ejemplos. Los procedimientos válidos (*workflows*: nombre, descripción y pasos ordenados) viven en **ChromaDB** como *embeddings*.
* **Búsqueda semántica:** se recupera el workflow con mayor **similitud coseno** al prompt.
* **Mistral 7B (Ollama, on-premises)** solo **extrae los argumentos** (`H1`, `H2`, `1`), con salida JSON validada.

```text
workflow «establish_ip_connection»
  paso 1: compute the shortest route ...            -> compute_route(src_node, dst_node)
  paso 2: provision and bring up a new IP connection -> establish_ip_connection(src_node, dst_node, bitrate_gbps)
```

> ⚙️ **Código de soporte (celdas plegadas).** Las siguientes celdas escriben/arrancan los componentes de esta estación: `workflows.json (4 workflows soportados)` · `knowledge_base.py (ChromaDB + embeddings)` · `llm_planner.py (planificador ChromaDB + Mistral 7B)`.
> **Solo ejecútelas** (Shift+Enter); no es necesario leerlas en clase. Puede desplegarlas para ver el detalle completo.

In [ ]:
%%writefile workflows.json
{
  "workflows": [
    {
      "name": "establish_ip_connection",
      "description": "Establish, create or provision a new IP connection (connectivity service) from a source node to a destination node with a requested bandwidth or bitrate.",
      "defaults": {},
      "steps": [
        {
          "intent": "compute the shortest route between source and destination nodes for a candidate connection",
          "tool": "compute_route",
          "args": ["src_node", "dst_node"]
        },
        {
          "intent": "provision and bring up a new IP connection on the SDN-controlled network",
          "tool": "establish_ip_connection",
          "args": ["src_node", "dst_node", "bitrate_gbps"]
        }
      ]
    },
    {
      "name": "remove_ip_connection",
      "description": "Remove, delete, tear down or release an existing IP connection between two nodes and restore isolation.",
      "defaults": {},
      "steps": [
        {
          "intent": "tear down an existing IP connection on the SDN-controlled network",
          "tool": "remove_ip_connection",
          "args": ["src_node", "dst_node"]
        }
      ]
    },
    {
      "name": "query_connection_status",
      "description": "Show, list or check the status and state of the IP connections configured in the SDN controller.",
      "defaults": {},
      "steps": [
        {
          "intent": "retrieve the status of IP connections from the SDN controller",
          "tool": "get_connection_status",
          "args": []
        }
      ]
    },
    {
      "name": "compute_route_only",
      "description": "Compute or find the shortest route or path between two nodes without provisioning anything (what-if path query).",
      "defaults": {},
      "steps": [
        {
          "intent": "compute the shortest route between source and destination nodes for a candidate connection",
          "tool": "compute_route",
          "args": ["src_node", "dst_node"]
        }
      ]
    }
  ]
}


In [ ]:
%%writefile knowledge_base.py
"""
knowledge_base.py
=================
Base de conocimiento vectorial de *workflows* (diseño Zero-Shot del paper).

  1. Un archivo de configuración (workflows.json) define los workflows
     soportados: ``name``, ``description`` y una secuencia ordenada de ``steps``.
  2. Cada workflow se convierte en *embedding* y se almacena en ChromaDB.
  3. El prompt del operador se convierte en embedding y se recupera el workflow
     más similar (similitud coseno). El prompt NO lleva ejemplos (zero-shot).

Backends de embeddings (``EMBEDDING_BACKEND``):
  * "ollama"         -> modelo de embeddings local en Ollama (p.ej. nomic-embed-text)
  * "chroma-default" -> all-MiniLM-L6-v2 (ONNX) de ChromaDB (descarga ~80 MB)
  * "hashing"        -> embedding léxico determinista, 100 % offline (respaldo)
  * "auto"           -> ollama si el modelo está disponible; si no, hashing
"""
from __future__ import annotations

import hashlib
import json
import math
import re
from typing import Any

import requests

try:
    import chromadb
    CHROMADB_AVAILABLE = True
except ImportError:  # pragma: no cover
    CHROMADB_AVAILABLE = False

STOPWORDS = {"a", "an", "the", "of", "to", "from", "with", "and", "or", "for", "on", "in", "into",
             "between", "by", "please", "new", "existing", "network", "nodes", "node", "any",
             "without", "anything", "two", "is", "are", "be", "that", "this", "it", "me", "i"}

CANONICAL = {
    "establish": "establish", "create": "establish", "provision": "establish", "setup": "establish",
    "set": "establish", "bring": "establish", "connect": "establish", "open": "establish",
    "remove": "remove", "delete": "remove", "tear": "remove", "teardown": "remove",
    "release": "remove", "terminate": "remove", "destroy": "remove",
    "route": "route", "path": "route", "routing": "route", "shortest": "route", "find": "route",
    "status": "status", "state": "status", "show": "status", "list": "status", "check": "status",
    "connection": "connection", "connectivity": "connection", "circuit": "connection", "service": "connection",
    "bandwidth": "bandwidth", "bitrate": "bandwidth", "capacity": "bandwidth", "gb": "bandwidth",
    "gbps": "bandwidth", "mbps": "bandwidth", "1gb": "bandwidth",
}


def _tokens(text: str) -> list[str]:
    out = []
    for tok in re.findall(r"[a-zA-Z][a-zA-Z0-9]*", text.lower()):
        if re.fullmatch(r"[a-z]\d+", tok) or tok in STOPWORDS:
            continue  # descarta identificadores de nodos (h1, r2...) y palabras vacías
        if tok.endswith("ing") and len(tok) > 5:
            tok = tok[:-3]
        elif tok.endswith("s") and len(tok) > 4:
            tok = tok[:-1]
        out.append(CANONICAL.get(tok, tok))
    return out


class HashingEmbedder:
    """Embedding léxico-semántico determinista (bolsa de palabras normalizadas + trigramas)."""

    name = "hashing-bow-512"

    def __init__(self, dim: int = 512):
        self.dim = dim

    def _bucket(self, feature: str) -> tuple[int, float]:
        h = int(hashlib.md5(feature.encode()).hexdigest(), 16)
        return h % self.dim, (1.0 if (h >> 64) & 1 else -1.0)

    def __call__(self, texts: list[str]) -> list[list[float]]:
        vectors = []
        for text in texts:
            vec = [0.0] * self.dim
            for tok in _tokens(text):
                idx, sign = self._bucket("w:" + tok)
                vec[idx] += sign
                for i in range(len(tok) - 2):
                    idx, sign = self._bucket("c:" + tok[i:i + 3])
                    vec[idx] += 0.2 * sign
            norm = math.sqrt(sum(v * v for v in vec)) or 1.0
            vectors.append([v / norm for v in vec])
        return vectors


class OllamaEmbedder:
    def __init__(self, host: str, model: str):
        self.host, self.model = host.rstrip("/"), model
        self.name = f"ollama:{model}"

    def __call__(self, texts: list[str]) -> list[list[float]]:
        r = requests.post(f"{self.host}/api/embed", json={"model": self.model, "input": texts}, timeout=120)
        r.raise_for_status()
        return r.json()["embeddings"]


class ChromaDefaultEmbedder:
    name = "chroma-default:all-MiniLM-L6-v2"

    def __init__(self):
        from chromadb.utils.embedding_functions import DefaultEmbeddingFunction
        self._ef = DefaultEmbeddingFunction()

    def __call__(self, texts: list[str]) -> list[list[float]]:
        return [list(map(float, v)) for v in self._ef(texts)]


def _ollama_has_model(host: str, model: str) -> bool:
    try:
        tags = requests.get(f"{host.rstrip('/')}/api/tags", timeout=2).json().get("models", [])
    except Exception:
        return False
    base = model.split(":")[0]
    return any(m["name"] == model or m["name"].split(":")[0] == base for m in tags)


def make_embedder(backend: str = "auto", ollama_host: str = "http://127.0.0.1:11434",
                  ollama_embed_model: str = "nomic-embed-text"):
    if backend == "ollama" or (backend == "auto" and _ollama_has_model(ollama_host, ollama_embed_model)):
        return OllamaEmbedder(ollama_host, ollama_embed_model)
    if backend == "chroma-default":
        return ChromaDefaultEmbedder()
    return HashingEmbedder()


class _InMemoryCollection:
    """Respaldo mínimo con la misma interfaz usada de ChromaDB (si no está instalado)."""

    def __init__(self):
        self.items: dict[str, tuple[list[float], str, dict]] = {}

    def upsert(self, ids, embeddings, documents, metadatas):
        for i, e, d, m in zip(ids, embeddings, documents, metadatas):
            self.items[i] = (e, d, m)

    def count(self) -> int:
        return len(self.items)

    def query(self, query_embeddings, n_results):
        q = query_embeddings[0]
        scored = sorted(((1 - sum(a * b for a, b in zip(q, e)), i, d, m)
                         for i, (e, d, m) in self.items.items()))[:n_results]
        return {"ids": [[s[1] for s in scored]], "distances": [[s[0] for s in scored]],
                "documents": [[s[2] for s in scored]], "metadatas": [[s[3] for s in scored]]}


class WorkflowKnowledgeBase:
    def __init__(self, workflows: list[dict], embedder=None, collection_name: str = "telecom_workflows"):
        self.embedder = embedder or HashingEmbedder()
        if CHROMADB_AVAILABLE:
            self.client = chromadb.EphemeralClient()
            try:
                self.client.delete_collection(collection_name)
            except Exception:
                pass
            self.collection = self.client.create_collection(
                collection_name, embedding_function=None, metadata={"hnsw:space": "cosine"})
            self.backend = f"ChromaDB {chromadb.__version__}"
        else:
            self.collection = _InMemoryCollection()
            self.backend = "in-memory (ChromaDB no instalado)"
        for wf in workflows:
            self.add_workflow(wf)

    @classmethod
    def from_file(cls, path: str = "workflows.json", embedder=None) -> "WorkflowKnowledgeBase":
        with open(path, encoding="utf-8") as fh:
            return cls(json.load(fh)["workflows"], embedder)

    @staticmethod
    def document(wf: dict) -> str:
        intents = "; ".join(s["intent"] for s in wf["steps"])
        return f"{wf['name'].replace('_', ' ')}: {wf['description']} Steps: {intents}"

    def add_workflow(self, wf: dict) -> None:
        doc = self.document(wf)
        self.collection.upsert(
            ids=[wf["name"]], embeddings=self.embedder([doc]), documents=[doc],
            metadatas=[{"name": wf["name"], "description": wf["description"],
                        "steps": json.dumps(wf["steps"]), "defaults": json.dumps(wf.get("defaults", {}))}])

    def search(self, prompt: str, k: int = 3) -> list[dict[str, Any]]:
        k = min(k, self.collection.count())
        res = self.collection.query(query_embeddings=self.embedder([prompt]), n_results=k)
        hits = []
        for dist, meta in zip(res["distances"][0], res["metadatas"][0]):
            hits.append({"name": meta["name"], "description": meta["description"],
                         "steps": json.loads(meta["steps"]), "defaults": json.loads(meta["defaults"]),
                         "similarity": round(1.0 - float(dist), 4)})
        return hits


In [ ]:
%%writefile llm_planner.py
"""
llm_planner.py
==============
Servicio "LLM" del Orquestador (Mensaje 2 de la Fig. 3: ``POST /plan``).

    prompt ──► ChromaDB (workflow más similar) ──► Mistral 7B @ Ollama
           (extrae SOLO los argumentos) ──► validación determinista ──► plan JSON

Si Ollama no está disponible se usa ``RuleBasedExtractor`` (LLM emulado) con
exactamente la misma interfaz, de modo que el notebook funciona en cualquier PC.
"""
import json
import re
from typing import Any

import requests

import workshop_common as wc

try:
    import ollama as ollama_lib
except ImportError:  # pragma: no cover
    ollama_lib = None

# El prompt de sistema vive en workshop_common para que el precalentamiento use los mismos tokens
EXTRACTION_SYSTEM_PROMPT = wc.EXTRACTION_SYSTEM_PROMPT
PARAM_MEANINGS = wc.PARAM_MEANINGS


def _normalize_number(value: Any) -> int | float:
    if isinstance(value, str):
        m = re.search(r"(\d+(?:\.\d+)?)\s*([tgmk])?", value.lower())
        if not m:
            raise ValueError(f"Valor numérico inválido: {value!r}")
        scale = {"t": 1000, "g": 1, "m": 1e-3, "k": 1e-6, None: 1}[m.group(2)]
        value = float(m.group(1)) * scale
    value = float(value)
    return int(value) if value.is_integer() else value


class RuleBasedExtractor:
    """'LLM emulado': extracción por expresiones regulares (determinista)."""

    NODE = r"([A-Za-z]+\d+)"
    PATTERNS = {
        "src_node": [rf"\bfrom\s+(?:node\s+)?{NODE}", rf"\bbetween\s+(?:node\s+)?{NODE}"],
        "dst_node": [rf"\bfrom\s+(?:node\s+)?[A-Za-z]+\d+\s+to\s+(?:node\s+)?{NODE}",
                     rf"\bbetween\s+(?:node\s+)?[A-Za-z]+\d+\s+and\s+(?:node\s+)?{NODE}"],
        "bitrate_gbps": [r"(\d+(?:\.\d+)?\s*[tgm])(?:b|bit|bits|bps|b/s)?\b"],
        "leaf_node": [r"\b((?:ONU|LEAF|L)\d+)\b"],
        "num_sc": [r"(\d+)\s*(?:digital\s+)?(?:sub-?carriers?|SCs?)\b"],
    }

    def extract(self, prompt: str, keys: list[str]) -> dict:
        out = {}
        for key in keys:
            for pattern in self.PATTERNS.get(key, []):
                m = re.search(pattern, prompt, flags=re.IGNORECASE)
                if m:
                    out[key] = m.group(1)
                    break
        return out


class LocalLLM:
    """Cliente del LLM on-premises (Mistral 7B servido por Ollama)."""

    def __init__(self, host: str = "http://127.0.0.1:11434", model: str = "mistral:7b",
                 enabled: Any = "auto"):
        self.host, self.requested_model = host.rstrip("/"), model
        self.model = model
        self.available = False
        if enabled is False:
            self.reason = "deshabilitado por configuración (USE_OLLAMA=False)"
            return
        try:
            tags = requests.get(f"{self.host}/api/tags", timeout=3).json().get("models", [])
        except Exception as exc:
            self.reason = f"Ollama no responde en {self.host} ({type(exc).__name__})"
            return
        names = [m["name"] for m in tags]
        base = model.split(":")[0]
        if model in names:
            self.model = model
        else:
            # Acepta otras etiquetas del mismo modelo, p.ej. mistral:latest (= Mistral 7B)
            alt = [n for n in names if n.split(":")[0] == base]
            if not alt:
                self.reason = f"modelo '{model}' no descargado (ejecute: ollama pull {model})"
                return
            self.model = alt[0]
        self.available = True
        self.reason = f"Ollama OK - modelo {self.model}"

    def chat_json(self, system: str, user: str, timeout: float = 600.0) -> dict:
        messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        options = dict(wc.LLM_OPTIONS)
        if ollama_lib is not None:
            client = ollama_lib.Client(host=self.host, timeout=timeout)
            resp = client.chat(model=self.model, messages=messages, format="json", options=options,
                               keep_alive=wc.LLM_KEEP_ALIVE)
            content = resp["message"]["content"]
        else:
            r = requests.post(f"{self.host}/api/chat", timeout=timeout,
                              json={"model": self.model, "messages": messages, "format": "json",
                                    "stream": False, "options": options, "keep_alive": wc.LLM_KEEP_ALIVE})
            r.raise_for_status()
            content = r.json()["message"]["content"]
        return json.loads(content)


class Planner:
    """Convierte un prompt en lenguaje natural en el plan estructurado (Fig. 5)."""

    def __init__(self, kb, llm: LocalLLM, known_nodes: set[str] | None = None, min_similarity: float = 0.15):
        self.kb, self.llm = kb, llm
        self.rules = RuleBasedExtractor()
        self.known_nodes = set(known_nodes or [])
        self.min_similarity = min_similarity
        self.last_retrieval: list[dict] = []
        self.last_backend = ""
        self.last_raw: Any = None

    def _validate(self, raw: dict, keys: list[str], defaults: dict) -> dict:
        params = {}
        for key in keys:
            value = raw.get(key)
            if value is None or (isinstance(value, str) and value.strip().lower() in ("", "null", "none")):
                value = defaults.get(key)   # el prompt no lo menciona -> valor por defecto del workflow
            if value is None or value == "":
                raise ValueError(f"Falta el parámetro '{key}'")
            if key.endswith("_node"):
                value = str(value).strip().upper().replace(" ", "")
                if self.known_nodes and value not in self.known_nodes:
                    raise ValueError(f"Nodo desconocido '{value}'")
            elif key == "num_sc":
                value = int(_normalize_number(value))
            elif key.endswith("_gbps"):
                value = _normalize_number(value)
                if value <= 0:
                    if key not in defaults:
                        raise ValueError(f"'{key}' debe ser > 0")
                    value = defaults[key]
            params[key] = value
        return params

    def extract(self, prompt: str, keys: list[str], defaults: dict) -> dict:
        if not keys:
            self.last_backend = "sin argumentos"
            return {}
        if self.llm.available:
            system = wc.extraction_system_prompt(keys)
            try:
                self.last_raw = self.llm.chat_json(system, prompt)
                params = self._validate(self.last_raw, keys, defaults)
                self.last_backend = f"ollama:{self.llm.model}"
                return params
            except Exception as exc:  # guardarraíl: salida del LLM inválida -> respaldo
                print(f"[Planner] Salida del LLM descartada ({exc}); usando extractor determinista")
        self.last_raw = self.rules.extract(prompt, keys)
        self.last_backend = "LLM emulado (reglas)"
        return self._validate(self.last_raw, keys, defaults)

    def plan(self, prompt: str) -> dict:
        self.last_retrieval = self.kb.search(prompt, k=3)
        best = self.last_retrieval[0]
        if best["similarity"] < self.min_similarity:
            raise ValueError(f"Ningún workflow soportado coincide con el prompt (sim={best['similarity']})")
        keys: list[str] = []
        for step in best["steps"]:
            keys += [a for a in step["args"] if a not in keys]
        params = self.extract(prompt, keys, best.get("defaults", {}))
        return {
            "steps": [{"intent": s["intent"]} for s in best["steps"]],
            "plans": [{"tool": s["tool"], "args": {a: params[a] for a in s["args"]}, "source": "llm"}
                      for s in best["steps"]],
        }


def create_llm_app(planner: Planner):
    from fastapi import FastAPI, HTTPException
    from pydantic import BaseModel

    class PromptRequest(BaseModel):
        prompt: str

    app = FastAPI(title="LLM planner (ChromaDB + Ollama)")

    @app.post("/plan")
    def plan(req: PromptRequest):
        try:
            return planner.plan(req.prompt)
        except ValueError as exc:
            raise HTTPException(status_code=422, detail=str(exc))

    @app.get("/health")
    def health():
        return {"llm": planner.llm.reason, "kb": planner.kb.backend,
                "embedder": planner.kb.embedder.name}

    return app


In [ ]:
# ▶️ ChromaDB: ¿qué workflow recupera cada frase?
knowledge_base = load_module("knowledge_base")
embedder = knowledge_base.make_embedder(wc.CONFIG["EMBEDDING_BACKEND"], OLLAMA_HOST, wc.CONFIG["OLLAMA_EMBED_MODEL"])
kb = knowledge_base.WorkflowKnowledgeBase.from_file("workflows.json", embedder)
print(f"Base vectorial: {kb.backend} · embeddings: {embedder.name} · workflows: {kb.collection.count()}")
viz.plot_similarity(kb, [PROMPT, "Remove the IP connection between H1 and H2",
                         "Show me the status of the connections", "Find the shortest path between H1 and H2"])

In [ ]:
# ▶️ Del prompt al plan (Mensaje 2 de la Figura 5). Con Mistral en CPU tarda ~10-50 s (ya precalentado en la Estación 0)
llm_planner = load_module("llm_planner")
llm = llm_planner.LocalLLM(OLLAMA_HOST, OLLAMA_MODEL, enabled=wc.CONFIG["USE_OLLAMA"])
planner = llm_planner.Planner(kb, llm, known_nodes=set(TOPOLOGY["hosts"]) | set(TOPOLOGY["switches"]))
LLM_URL = wc.start_server("llm", llm_planner.create_llm_app(planner), "llm")   # servicio POST /plan

t0 = time.time()
plan_preview = planner.plan(PROMPT)
print(f"🧠 LLM: {llm.reason}")
print(f"🔎 Workflow recuperado: {planner.last_retrieval[0]['name']}")
print(f"🧩 Argumentos extraídos ({planner.last_backend}, {time.time() - t0:.1f} s): {planner.last_raw}")
print(wc.pretty(plan_preview))

👀 **Observe:** `steps` viene del workflow (el **qué**), `plans` trae herramienta + argumentos (el **con qué**). El LLM no decidió la ruta ni las reglas: solo leyó `H1`, `H2` y `1GB → 1`.

💬 **Pregunta rápida:** ¿qué pasaría si el operador pide algo que **no** está en ningún workflow?

---
## 🟦🟦🟦🟦🟦⬜ Estación 4 · Agentes A2A y el flujo completo de 8 pasos · ⏱️ 12 min
🎯 **Objetivo:** reproducir la demostración del paper de punta a punta y **auditar** cada mensaje (Figuras 3, 4 y 5).

📖 **Concepto en 1 minuto** *(Diapositivas 7 y 11)*
* **A2A (Agent2Agent):** cada agente publica su **Agent Card** (`/.well-known/agent-card.json`) con sus *skills*. El Orquestador hace **Handoff**: delega el plan al agente cuyas skills cubren las herramientas pedidas. La delegación es una **Task** con estados (`WORKING → COMPLETED`, o `INPUT_REQUIRED` si se pide aprobación humana).
* **Flujo:** ① `POST /prompt` → ② `POST /plan` → ③ `POST /execute_plan` (A2A) → ④ `POST /compute_route` (MCP→PCE) → ⑤ `POST /service/ip/create` (MCP→Ryu) → ⑥ `OFPT_FLOW_MOD` → ⑦ y ⑧ `200 OK` con la traza.

> ⚙️ **Código de soporte (celdas plegadas).** Las siguientes celdas escriben/arrancan los componentes de esta estación: `a2a_agents.py (IPConnectionAgent A2A)` · `orchestrator.py (Orquestador centralizado)` · `arranque del agente y del orquestador`.
> **Solo ejecútelas** (Shift+Enter); no es necesario leerlas en clase. Puede desplegarlas para ver el detalle completo.

In [ ]:
%%writefile a2a_agents.py
"""
a2a_agents.py
=============
Agentes especializados interoperables vía A2A (Agent2Agent, https://a2a-protocol.org).

  * ``SpecializedAgent``: clase base reutilizable (Agent Card, ciclo de vida de
    Task, sesión MCP, asociación paso->herramienta, Human-in-the-Loop).
  * ``IPConnectionAgent``: agente de la capa de paquetes del paper.
  * ``create_agent_app``: *bindings* HTTP del agente:
        GET  /.well-known/agent-card.json   (A2A v1.0; alias legado /.well-known/agent.json)
        POST /execute_plan                  (binding usado en el paper, Fig. 3)
        POST /a2a                           (JSON-RPC 2.0: SendMessage, GetTask)
"""
import asyncio
import copy
import datetime as dt
import uuid
from typing import Any

import workshop_common as wc
from knowledge_base import HashingEmbedder
from mcp_server import MCPSession

COMPLETED, FAILED = "TASK_STATE_COMPLETED", "TASK_STATE_FAILED"
WORKING, INPUT_REQUIRED, SUBMITTED = "TASK_STATE_WORKING", "TASK_STATE_INPUT_REQUIRED", "TASK_STATE_SUBMITTED"
CANCELED = "TASK_STATE_CANCELED"
TERMINAL_STATES = {COMPLETED, FAILED, CANCELED, "TASK_STATE_REJECTED"}

TOOL_MAPPING_PROMPT = (
    "You map a network operation step to exactly one tool. Available tools:\n{tools}\n"
    "Reply ONLY with JSON: {{\"tool\": \"<tool name>\"}}"
)


def _now() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")


def _cosine(a: list[float], b: list[float]) -> float:
    return sum(x * y for x, y in zip(a, b))


class SpecializedAgent:
    """Agente especializado A2A que ejecuta planes invocando herramientas MCP."""

    name = "SpecializedAgent"
    description = ""
    version = "1.0.0"
    skills: list[dict] = []
    allowed_tools: set[str] | None = None     # None = todas las herramientas MCP
    hitl_after_tools: set[str] = set()        # pausa HITL tras estas herramientas

    def __init__(self, base_url: str = "", llm: Any = None, hitl: bool = False,
                 mcp_factory: Any = MCPSession, label: str = "Agent"):
        self.base_url = base_url
        self.llm = llm
        self.hitl = hitl
        self.mcp_factory = mcp_factory
        self.label = label
        self.tasks: dict[str, dict] = {}
        self._plans: dict[str, dict] = {}
        self._embed = HashingEmbedder()
        self.last_associations: list[dict] = []

    # ------------------------------------------------------------------ Agent Card
    def agent_card(self) -> dict:
        return {
            "name": self.name,
            "description": self.description,
            "version": self.version,
            "provider": {"organization": "Workshop Redes de Acceso", "url": self.base_url},
            "supportedInterfaces": [
                {"url": f"{self.base_url}/a2a", "protocolBinding": "JSONRPC", "protocolVersion": "1.0"},
                {"url": f"{self.base_url}/execute_plan", "protocolBinding": "HTTP+JSON", "protocolVersion": "1.0"},
            ],
            "capabilities": {"streaming": False, "pushNotifications": False},
            "defaultInputModes": ["application/json", "text/plain"],
            "defaultOutputModes": ["application/json"],
            "skills": self.skills,
        }

    def supported_tools(self) -> set[str]:
        return {t.split(":", 1)[1] for s in self.skills for t in s.get("tags", []) if t.startswith("tool:")}

    # ------------------------------------------------------------------ Task lifecycle
    def _new_task(self, plan: dict) -> dict:
        task = {"id": str(uuid.uuid4()), "contextId": str(uuid.uuid4()),
                "status": {"state": SUBMITTED, "timestamp": _now()},
                "artifacts": [], "history": [], "metadata": {"agent": self.name}}
        self.tasks[task["id"]] = task
        self._plans[task["id"]] = {"plan": copy.deepcopy(plan), "cursor": 0, "trace": []}
        return task

    def _set_state(self, task: dict, state: str, text: str | None = None, data: Any = None) -> None:
        status: dict[str, Any] = {"state": state, "timestamp": _now()}
        if text or data is not None:
            parts: list[dict] = []
            if text:
                parts.append({"text": text})
            if data is not None:
                parts.append({"data": data})
            status["message"] = {"messageId": str(uuid.uuid4()), "role": "ROLE_AGENT",
                                 "taskId": task["id"], "contextId": task["contextId"], "parts": parts}
        task["status"] = status
        task["history"].append({"state": state, "timestamp": status["timestamp"]})

    # ------------------------------------------------------------------ Paso -> herramienta
    def _associate_semantic(self, intent: str, tools: dict[str, dict]) -> str:
        q = self._embed([intent])[0]
        scored = [(_cosine(q, self._embed([f"{n.replace('_', ' ')}: {t['description']}"])[0]), n)
                  for n, t in tools.items()]
        return max(scored)[1]

    def _associate_llm(self, intent: str, tools: dict[str, dict]) -> str | None:
        listing = "\n".join(f"- {n}: {t['description']}" for n, t in tools.items())
        try:
            answer = self.llm.chat_json(TOOL_MAPPING_PROMPT.format(tools=listing), f"Step: {intent}")
            tool = str(answer.get("tool", "")).strip()
            return tool if tool in tools else None
        except Exception:
            return None

    async def associate_tool(self, intent: str, planned_tool: str | None, tools: dict[str, dict]) -> str:
        """Asocia un paso del plan con una herramienta MCP (LLM si está disponible)."""
        method, chosen = "semantic", None
        if self.llm is not None and getattr(self.llm, "available", False):
            chosen = await asyncio.to_thread(self._associate_llm, intent, tools)
            method = f"llm:{self.llm.model}" if chosen else "semantic"
        if chosen is None:
            chosen = self._associate_semantic(intent, tools)
        if planned_tool and planned_tool not in tools:
            raise ValueError(f"La herramienta '{planned_tool}' no está disponible para {self.name}")
        final = planned_tool or chosen
        self.last_associations.append({"intent": intent, "planned": planned_tool, "associated": chosen,
                                       "method": method, "used": final, "agree": chosen == planned_tool})
        return final

    @staticmethod
    def validate_args(args: dict, schema: dict) -> dict:
        """Validación determinista de argumentos contra el JSON Schema de la herramienta MCP."""
        props, required = schema.get("properties", {}), schema.get("required", [])
        missing = [r for r in required if r not in args]
        if missing:
            raise ValueError(f"Faltan argumentos requeridos: {missing}")
        unknown = [a for a in args if a not in props]
        if unknown:
            raise ValueError(f"Argumentos no definidos en el esquema: {unknown}")
        for key, value in args.items():
            expected = props[key].get("type")
            if expected == "string" and not isinstance(value, str):
                raise ValueError(f"'{key}' debe ser string")
            if expected in ("number", "integer") and not isinstance(value, (int, float)):
                raise ValueError(f"'{key}' debe ser numérico")
        return args

    # ------------------------------------------------------------------ Ejecución
    async def _run(self, task: dict) -> dict:
        state = self._plans[task["id"]]
        plan = state["plan"]
        self._set_state(task, WORKING)
        try:
            async with self.mcp_factory(caller=self.label) as session:
                tools = {t["name"]: t for t in await session.list_tools()}
                if self.allowed_tools is not None:
                    tools = {n: t for n, t in tools.items() if n in self.allowed_tools}
                while state["cursor"] < len(plan["plans"]):
                    idx = state["cursor"]
                    step, item = plan["steps"][idx], plan["plans"][idx]
                    tool = await self.associate_tool(step["intent"], item.get("tool"), tools)
                    args = self.validate_args(dict(item.get("args", {})), tools[tool]["inputSchema"])
                    result = await session.call_tool(tool, args)
                    state["trace"].append({"step": idx + 1, "intent": step["intent"], "tool": tool,
                                           "args": args, "agent": self.name, "result": result})
                    state["cursor"] += 1
                    if self.hitl and tool in self.hitl_after_tools and state["cursor"] < len(plan["plans"]):
                        nxt = plan["plans"][state["cursor"]]
                        self._set_state(task, INPUT_REQUIRED,
                                        text=f"HITL: ¿Aprueba ejecutar '{nxt['tool']}' con {nxt['args']} "
                                             f"usando el resultado anterior? Responda {{'approved': true|false}}",
                                        data={"pending_step": state["cursor"] + 1, "last_result": result})
                        return task
        except Exception as exc:
            self._set_state(task, FAILED, text=f"{type(exc).__name__}: {exc}",
                            data={"trace": state["trace"]})
            return task
        task["artifacts"] = [{"artifactId": str(uuid.uuid4()), "name": "execution-trace",
                              "parts": [{"data": {"trace": state["trace"]}}]}]
        self._set_state(task, COMPLETED)
        return task

    async def execute_plan(self, plan: dict) -> dict:
        self.last_associations = []
        return await self._run(self._new_task(plan))

    async def resume(self, task_id: str, approved: bool) -> dict:
        task = self.tasks[task_id]
        if task["status"]["state"] != INPUT_REQUIRED:
            raise ValueError(f"La tarea {task_id} no espera input (estado {task['status']['state']})")
        if not approved:
            self._set_state(task, CANCELED, text="Operación rechazada por el operador (HITL)")
            return task
        return await self._run(task)


class IPConnectionAgent(SpecializedAgent):
    name = "IPConnectionAgent"
    description = ("Specialized packet-layer agent: computes routes with the PCE and provisions, "
                   "removes or audits IP connections through the SDN controller NBI exposed via MCP.")
    allowed_tools = {"compute_route", "establish_ip_connection", "remove_ip_connection", "get_connection_status"}
    hitl_after_tools = {"compute_route"}
    skills = [
        {"id": "ip-connection-provisioning", "name": "IP connection provisioning",
         "description": "Compute a route and establish an IP connection between two nodes with a bitrate.",
         "tags": ["ip", "sdn", "packet", "tool:compute_route", "tool:establish_ip_connection"],
         "examples": ["Establish an IP connection from H1 to H2 with 1GB bandwidth"],
         "inputModes": ["application/json"], "outputModes": ["application/json"]},
        {"id": "ip-connection-lifecycle", "name": "IP connection lifecycle",
         "description": "Remove IP connections and report their status.",
         "tags": ["ip", "sdn", "tool:remove_ip_connection", "tool:get_connection_status"],
         "examples": ["Remove the IP connection between H1 and H2"],
         "inputModes": ["application/json"], "outputModes": ["application/json"]},
    ]


def create_agent_app(agent: SpecializedAgent):
    from fastapi import FastAPI, HTTPException, Request

    app = FastAPI(title=f"A2A agent: {agent.name}")

    @app.get("/.well-known/agent-card.json")
    @app.get("/.well-known/agent.json")
    def agent_card():
        return agent.agent_card()

    @app.post("/execute_plan")
    async def execute_plan(plan: dict):
        if "plans" not in plan or "steps" not in plan:
            raise HTTPException(status_code=400, detail="El plan debe contener 'steps' y 'plans'")
        return await agent.execute_plan(plan)

    @app.post("/a2a")
    async def jsonrpc(request: Request):
        req = await request.json()
        rid, method, params = req.get("id"), req.get("method"), req.get("params", {})
        try:
            if method == "SendMessage":
                msg = params["message"]
                data = next((p["data"] for p in msg.get("parts", []) if "data" in p), {})
                if msg.get("taskId"):
                    result = await agent.resume(msg["taskId"], bool(data.get("approved")))
                else:
                    result = await agent.execute_plan(data)
            elif method == "GetTask":
                result = agent.tasks[params["id"]]
            else:
                return {"jsonrpc": "2.0", "id": rid, "error": {"code": -32601, "message": "Method not found"}}
        except (KeyError, ValueError) as exc:
            return {"jsonrpc": "2.0", "id": rid, "error": {"code": -32602, "message": str(exc)}}
        return {"jsonrpc": "2.0", "id": rid, "result": {"task": result}}

    return app


In [ ]:
%%writefile orchestrator.py
"""
orchestrator.py
===============
Agente Orquestador centralizado (Fig. 1 y 2 del paper):

  1. Recibe el prompt del operador            (POST /prompt)
  2. Pide el plan al servicio LLM             (POST /plan)
  3. Selecciona el agente especializado por sus *skills* (Agent Card A2A) y
     le delega el plan -> patrón **Handoff**  (POST /execute_plan)
  4. Devuelve al operador la traza de ejecución  {"trace": [...]}
"""
from typing import Any

import workshop_common as wc

COMPLETED, INPUT_REQUIRED = "TASK_STATE_COMPLETED", "TASK_STATE_INPUT_REQUIRED"


class Orchestrator:
    def __init__(self, llm_url: str):
        self.llm_url = llm_url
        self.agents: dict[str, dict[str, Any]] = {}
        self.pending_tasks: dict[str, str] = {}
        self.last_plan: dict | None = None
        self.last_handoff: str | None = None

    # ------------------------------------------------------------------ Descubrimiento A2A
    def register_agent(self, base_url: str, label: str = "Agent") -> dict:
        card = wc.http_call("Orchestrator", label, "GET", f"{base_url}/.well-known/agent-card.json")
        tools = {t.split(":", 1)[1] for s in card.get("skills", []) for t in s.get("tags", [])
                 if t.startswith("tool:")}
        self.agents[card["name"]] = {"card": card, "base_url": base_url, "label": label, "tools": tools}
        return card

    def select_agent(self, plan: dict) -> dict:
        """Handoff: elige el agente cuyas skills cubren todas las herramientas del plan."""
        needed = {p["tool"] for p in plan["plans"]}
        ranked = sorted(self.agents.values(), key=lambda a: len(needed & a["tools"]), reverse=True)
        if not ranked or not needed <= ranked[0]["tools"]:
            raise LookupError(f"Ningún agente registrado soporta las herramientas {sorted(needed)}")
        return ranked[0]

    # ------------------------------------------------------------------ Flujo principal
    def _finish(self, task: dict, agent: dict) -> dict:
        state = task["status"]["state"]
        if state == COMPLETED:
            self.pending_tasks.pop(task["id"], None)
            for artifact in task.get("artifacts", []):
                for part in artifact.get("parts", []):
                    if "trace" in part.get("data", {}):
                        return {"trace": part["data"]["trace"]}
            return {"trace": []}
        if state == INPUT_REQUIRED:
            self.pending_tasks[task["id"]] = agent["card"]["name"]
            parts = task["status"]["message"]["parts"]
            return {"status": state, "task_id": task["id"], "agent": agent["card"]["name"],
                    "question": next((p["text"] for p in parts if "text" in p), ""),
                    "details": next((p["data"] for p in parts if "data" in p), {})}
        msg = task["status"].get("message", {}).get("parts", [{}])[0].get("text", "")
        raise RuntimeError(f"El agente {agent['card']['name']} terminó en {state}: {msg}")

    def handle_prompt(self, prompt: str) -> dict:
        plan = wc.http_call("Orchestrator", "LLM", "POST", f"{self.llm_url}/plan", {"prompt": prompt})
        self.last_plan = plan
        agent = self.select_agent(plan)
        self.last_handoff = agent["card"]["name"]
        task = wc.http_call("Orchestrator", agent["label"], "POST", f"{agent['base_url']}/execute_plan", plan)
        return self._finish(task, agent)

    def approve(self, task_id: str, approved: bool) -> dict:
        agent = self.agents[self.pending_tasks[task_id]]
        rpc = {"jsonrpc": "2.0", "id": 1, "method": "SendMessage",
               "params": {"message": {"messageId": task_id + "-hitl", "role": "ROLE_USER", "taskId": task_id,
                                      "parts": [{"data": {"approved": approved}}]}}}
        resp = wc.http_call("Orchestrator", agent["label"], "POST", f"{agent['base_url']}/a2a", rpc)
        if "error" in resp:
            raise RuntimeError(resp["error"]["message"])
        task = resp["result"]["task"]
        if task["status"]["state"] == "TASK_STATE_CANCELED":
            self.pending_tasks.pop(task_id, None)
            return {"status": "TASK_STATE_CANCELED", "task_id": task_id}
        return self._finish(task, agent)


def create_orchestrator_app(orch: Orchestrator):
    from fastapi import FastAPI, HTTPException
    from pydantic import BaseModel

    class PromptRequest(BaseModel):
        prompt: str

    class ApprovalRequest(BaseModel):
        task_id: str
        approved: bool

    app = FastAPI(title="Centralized Orchestrator")

    @app.post("/prompt")
    def prompt(req: PromptRequest):
        try:
            return orch.handle_prompt(req.prompt)
        except wc.HTTPCallError as exc:
            raise HTTPException(status_code=502, detail=exc.detail)
        except (LookupError, RuntimeError) as exc:
            raise HTTPException(status_code=502, detail=str(exc))

    @app.post("/approve")
    def approve(req: ApprovalRequest):
        try:
            return orch.approve(req.task_id, req.approved)
        except KeyError:
            raise HTTPException(status_code=404, detail=f"Tarea {req.task_id} no pendiente")
        except RuntimeError as exc:
            raise HTTPException(status_code=502, detail=str(exc))

    @app.get("/agents")
    def agents():
        return {name: {"url": a["base_url"], "tools": sorted(a["tools"])} for name, a in orch.agents.items()}

    return app


In [ ]:
# ⚙️ Arranque del agente especializado y del Orquestador + referencias del paper (Fig. 5)
a2a_agents = load_module("a2a_agents")
orchestrator_mod = load_module("orchestrator")
ip_agent = a2a_agents.IPConnectionAgent(llm=llm if AGENT_LLM_TOOL_MAPPING else None)
ip_agent.base_url = wc.start_server("agent", a2a_agents.create_agent_app(ip_agent), "agent")
orchestrator = orchestrator_mod.Orchestrator(LLM_URL)
_ = orchestrator.register_agent(ip_agent.base_url, label="Agent")       # descubrimiento A2A (Agent Card)
ORCH_URL = wc.start_server("orchestrator", orchestrator_mod.create_orchestrator_app(orchestrator), "orchestrator")


def paper_step(frame):
    # Asocia cada trama capturada con el paso (1-8) de la Figura 2 del paper
    s, d, info, proto = frame["src"], frame["dst"], frame["info"], frame["protocol"]
    if proto == "OpenFlow":
        return "6"
    rules = [("User", "Orchestrator", "/prompt", "1"), ("Orchestrator", "LLM", "/plan", "2"),
             ("LLM", "Orchestrator", "HTTP/1.1 2", "2"), ("Orchestrator", "Agent", "/execute_plan", "3"),
             ("Agent", "PCE", "/compute_route", "4"), ("PCE", "Agent", "HTTP/1.1 2", "4"),
             ("Agent", "SDN", "/service/ip/create", "5"), ("SDN", "Agent", "HTTP/1.1 2", "5"),
             ("Agent", "Orchestrator", "HTTP/1.1 2", "7"), ("Orchestrator", "User", "HTTP/1.1 2", "8")]
    return next((st for rs, rd, key, st in rules if s == rs and d == rd and key in info), "")


EXPECTED_PLAN = {
    "steps": [{"intent": "compute the shortest route between source and destination nodes for a candidate connection"},
              {"intent": "provision and bring up a new IP connection on the SDN-controlled network"}],
    "plans": [{"tool": "compute_route", "args": {"src_node": "H1", "dst_node": "H2"}, "source": "llm"},
              {"tool": "establish_ip_connection", "args": {"src_node": "H1", "dst_node": "H2", "bitrate_gbps": 1},
               "source": "llm"}]}
EXPECTED_TRACE = {"trace": [
    {"step": 1, "intent": EXPECTED_PLAN["steps"][0]["intent"], "tool": "compute_route",
     "args": {"src_node": "H1", "dst_node": "H2"}, "agent": "IPConnectionAgent",
     "result": {"src": "H1", "dst": "H2", "path": ["H1", "R1", "R2", "R3", "H2"], "hops": 4}},
    {"step": 2, "intent": EXPECTED_PLAN["steps"][1]["intent"], "tool": "establish_ip_connection",
     "args": {"src_node": "H1", "dst_node": "H2", "bitrate_gbps": 1}, "agent": "IPConnectionAgent",
     "result": {"connection_id": "H1-H2", "status": "UP", "message": "Connection H1-H2 established"}}]}
print(f"✅ IPConnectionAgent en {ip_agent.base_url} · Orquestador en {ORCH_URL}")

In [ ]:
# ▶️ La Agent Card A2A que el Orquestador leyó para poder delegar (resumen)
card = requests.get(f"{ip_agent.base_url}/.well-known/agent-card.json", timeout=10).json()
print(f"🤖 {card['name']} — {card['description']}")
for itf in card["supportedInterfaces"]:
    print(f"   interfaz {itf['protocolBinding']:<9} {itf['url']}")
for sk in card["skills"]:
    print(f"   skill '{sk['id']}': {[t for t in sk['tags'] if t.startswith('tool:')]}")

In [ ]:
# ▶️ 🚀 EL MOMENTO DEL PAPER: el operador envía su intención
if connection_path():                                     # partimos siempre del estado aislado
    requests.delete(f"{SDN_URL}/service/ip/delete", params={"connection_id": "H1-H2"}, timeout=10)

wc.CAPTURE.reset()
t0 = time.time()
response = wc.http_call("User", "Orchestrator", "POST", f"{ORCH_URL}/prompt", {"prompt": PROMPT})
print(f"🗣️  {PROMPT!r}")
print(f"⏱️  {time.time() - t0:.1f} s · 🧠 {planner.last_backend} · 🤝 Handoff A2A -> {orchestrator.last_handoff}")

In [ ]:
# ▶️ Figuras 2 y 3: ¿quién habló con quién? (diagrama generado desde la captura real)
frames = wc.CAPTURE.filter(("HTTP", "OpenFlow"))
viz.plot_sequence(frames, paper_step)

seen = {}
for f in frames:
    seen.setdefault(paper_step(f), []).append(f)
checks = [("1", "POST /prompt"), ("2", "POST /plan"), ("3", "POST /execute_plan (A2A)"),
          ("4", "POST /compute_route (MCP→PCE)"), ("5", "POST /service/ip/create (MCP→Ryu)"),
          ("6", "OFPT_FLOW_MOD (OpenFlow 1.3)"), ("7", "200 OK de /execute_plan"), ("8", "200 OK de /prompt")]
print("  ".join(f"{'✅' if (seen.get(s) or (s == '6' and not SIMULATION_MODE)) else '❌'} {s}·{t}" for s, t in checks[:4]))
print("  ".join(f"{'✅' if (seen.get(s) or (s == '6' and not SIMULATION_MODE)) else '❌'} {s}·{t}" for s, t in checks[4:]))
print("\nCaptura estilo Wireshark (solo HTTP; use wc.CAPTURE.show(protocols=None) para ver también MCP y OpenFlow):")
wc.CAPTURE.show(protocols=("HTTP",), annotate=paper_step)

In [ ]:
# ▶️ Figura 5: plan (mensaje 2) y traza (mensaje 8) comparados con el paper
plan_msg = next(f["payload"] for f in wc.CAPTURE.frames if f["src"] == "LLM" and f["dst"] == "Orchestrator")
print("Plan  (Mensaje 2):", "✅ idéntico al paper" if plan_msg == EXPECTED_PLAN else "❌ difiere")
print("Trace (Mensaje 8):", "✅ idéntico al paper" if response == EXPECTED_TRACE else "❌ difiere")
print(wc.pretty(response))

In [ ]:
# ▶️ Figura 4: el plano de datos después de la intención
mininet_ping("H1", "H2")
viz.plot_network(inventory, "Estación 4 · Conexión H1–H2 provisionada por el agente vía MCP → Ryu → OpenFlow",
                 connection_path=connection_path(), computed_path=response["trace"][0]["result"]["path"])
dump_flows("R2")

👀 **Observe:** de una sola frase salieron **12 reglas OpenFlow** (IPv4 + ARP, ida y vuelta, en R1–R3) y el ping ahora responde; el primer eco es más lento porque incluye la resolución ARP. Toda la decisión quedó **auditada**: plan, traza, mensajes HTTP y reglas.

💬 **Pregunta rápida:** ¿en qué paso intervendría un humano si activamos *Human-in-the-Loop*? (lo probaremos en la Estación 5, escenario ②)

---
## 🟦🟦🟦🟦🟦🟦 Estación 5 · Exploración interactiva · ⏱️ 20 min
🎯 **Objetivo:** probar libremente el sistema **tal cual está** y conectar cada concepto con lo que ocurre en la red.

📖 **Antes de abrir la consola** *(Diapositiva 16)*: precargamos dos **extensiones hacia las redes de acceso** que siguen exactamente el patrón del paper (*herramienta MCP + workflow en ChromaDB + agente A2A*):

| Extensión | Inspirada en | Qué agrega |
|---|---|---|
| 🔆 **Agente óptico P2MP** | Ref. [2] | Herramienta `allocate_subcarriers` (16 subportadoras × 25 Gb/s, First-Fit contiguo) y un **nuevo agente A2A** `OpticalP2MPAgent` |
| 🩺 **Gemelo digital** | Ref. [3] | Herramienta `get_e2e_telemetry` (latencia y BER) y el workflow *diagnosticar degradación → recalcular ruta → re-provisionar* (usa el **bypass óptico** R1–R3) |

### 🧪 Retos de exploración sugeridos (en este orden)
| # | En la consola | Observe |
|---|---|---|
| 1 | Escenario **①** y avance con ▶ | Los 8 pasos en la arquitectura animada; pestaña 🔀: la ruta y las reglas de R1, R2 y R3 |
| 2 | Escenario **②** con *Rechazar* y luego con *Aprobar* | La Task A2A queda en `INPUT_REQUIRED`; si rechaza, **no** se envía ningún `FLOW_MOD` |
| 3 | Escenarios **③**, **④** y **⑤** | Distinto workflow en 🧠; ④ no toca la red; ⑤ borra reglas y vuelve el aislamiento |
| 4 | Escenario **⑥** dos veces | **Handoff** al agente óptico; pestaña 🌈: cómo se llena el espectro del Hub |
| 5 | Escenario **⑦** | Enlace R2–R3 degradado → nueva ruta `H1-R1-R3-H2` por el bypass óptico |
| 6 | ✍️ **Prompt libre** | Invente frases: `Allocate 4 subcarriers to leaf ONU3 in the P2MP access tree`, `Establish an IP connection from H1 to H2 with 2 Gbps`… ¿qué workflow elige ChromaDB? |

> 💡 Por defecto la consola usa el LLM emulado (segundos por escenario). Marque **Usar Mistral 7B real** para ver el LLM local en acción.

> ⚙️ **Código de soporte (celdas plegadas).** Las siguientes celdas escriben/arrancan los componentes de esta estación: `access_extensions.py (extensiones P2MP y gemelo digital)` · `workshop_dashboard.py (consola interactiva)`.
> **Solo ejecútelas** (Shift+Enter); no es necesario leerlas en clase. Puede desplegarlas para ver el detalle completo.

In [ ]:
%%writefile access_extensions.py
"""
access_extensions.py
====================
Extensiones de la arquitectura hacia las REDES DE ACCESO (precargadas para la
exploración interactiva). Siguen exactamente el mismo patrón del paper:
herramienta MCP determinista + workflow en ChromaDB + agente A2A.

  * Ref. [2] · P2MP con subportadoras digitales (DSCM):
      - herramienta MCP  allocate_subcarriers(leaf_node, num_sc)
      - nuevo agente A2A OpticalP2MPAgent  (el Orquestador le hace Handoff)
  * Ref. [3] · Gemelo digital (OCATA simplificado):
      - herramienta MCP  get_e2e_telemetry(src_node, dst_node)
      - workflow «diagnose_latency_degradation»: telemetría -> ruta -> re-provisión
"""
import networkx as nx
import requests

import a2a_agents
import workshop_common as wc

HUB_SUBCARRIERS = 16    # el Hub recibe 16 subportadoras digitales
SC_RATE_GBPS = 25       # 25 Gb/s por subportadora (16 x 25 = 400 Gb/s)
LEAF_MAX_SC = 4         # cada hoja modula como máximo 4 subportadoras (100 Gb/s)
P2MP_LEAVES = ("ONU1", "ONU2", "ONU3", "ONU4", "ONU5")


class P2MPSubcarrierManager:
    """Asignación First-Fit de subportadoras CONTIGUAS sin solapamiento espectral."""

    def __init__(self, leaves=P2MP_LEAVES):
        self.leaves = set(leaves)
        self.slots = [None] * HUB_SUBCARRIERS

    def allocate_subcarriers(self, leaf_node: str, num_sc: int) -> dict:
        if leaf_node not in self.leaves:
            raise ValueError(f"{leaf_node} no pertenece al árbol P2MP")
        if not 1 <= num_sc <= LEAF_MAX_SC:
            raise ValueError(f"num_sc debe estar entre 1 y {LEAF_MAX_SC}")
        previous = list(self.slots)
        self.release(leaf_node)
        for start in range(HUB_SUBCARRIERS - num_sc + 1):
            block = range(start, start + num_sc)
            if all(self.slots[i] is None for i in block):
                for i in block:
                    self.slots[i] = leaf_node
                used = sum(s is not None for s in self.slots)
                return {"leaf_node": leaf_node, "subcarriers": list(block),
                        "capacity_gbps": num_sc * SC_RATE_GBPS, "hub_utilization": used / HUB_SUBCARRIERS}
        self.slots = previous
        raise ValueError(f"No hay {num_sc} subportadoras contiguas libres en el Hub")

    def release(self, leaf_node: str) -> None:
        self.slots = [None if s == leaf_node else s for s in self.slots]

    def spectrum_map(self) -> str:
        return "|" + "".join("·" if s is None else s[-1] for s in self.slots) + "|"


class OpticalP2MPAgent(a2a_agents.SpecializedAgent):
    name = "OpticalP2MPAgent"
    description = ("Specialized optical-layer agent for point-to-multipoint (P2MP) access trees based on "
                   "digital subcarrier multiplexing (DSCM): allocates 25 Gb/s subcarriers to leaf nodes.")
    allowed_tools = {"allocate_subcarriers"}
    skills = [
        {"id": "p2mp-subcarrier-allocation", "name": "P2MP subcarrier allocation",
         "description": "Allocate contiguous digital subcarriers to a leaf node without spectral overlap.",
         "tags": ["optical", "p2mp", "dscm", "access", "tool:allocate_subcarriers"],
         "examples": ["Allocate 2 subcarriers to leaf ONU1 in the P2MP access tree"],
         "inputModes": ["application/json"], "outputModes": ["application/json"]},
    ]


class OCATADigitalTwinMock:
    """Gemelo digital simplificado: estima latencia e2e y BER pre-FEC de la conexión provisionada."""

    BER_FEC_THRESHOLD = 4e-3

    def __init__(self, inventory, d_max_ms=1.5):
        self.inventory, self.d_max_ms = inventory, d_max_ms
        self.impairments = {}

    def inject_degradation(self, u, v, extra_ms=2.0, ber=6e-3):
        self.impairments[frozenset((u, v))] = {"extra_ms": extra_ms, "ber": ber}
        self.inventory.set_link_state(u, v, "degraded")      # alarma -> el PCE evitará el enlace

    def clear(self):
        for key in list(self.impairments):
            u, v = tuple(key)
            self.inventory.set_link_state(u, v, "up")
        self.impairments.clear()

    def _provisioned_path(self, src, dst):
        conns = requests.get(f"{wc.url('sdn')}/service/ip/status", timeout=10).json().get("connections", {})
        if f"{src}-{dst}" in conns:
            return conns[f"{src}-{dst}"]["path"]
        return nx.shortest_path(self.inventory.graph, src, dst)

    def estimate(self, src, dst):
        path = self._provisioned_path(src, dst)
        latency, ber = 0.0, 1e-6
        for u, v in zip(path, path[1:]):
            imp = self.impairments.get(frozenset((u, v)), {})
            latency += self.inventory.graph.edges[u, v]["delay_ms"] + imp.get("extra_ms", 0.0)
            ber = max(ber, imp.get("ber", 1e-6))
        return {"src": src, "dst": dst, "path": path, "latency_ms": round(latency, 3),
                "d_max_ms": self.d_max_ms, "pre_fec_ber": ber,
                "qot_ok": ber < self.BER_FEC_THRESHOLD, "degraded": latency > self.d_max_ms}


DIAGNOSE_WORKFLOW = {
    "name": "diagnose_latency_degradation",
    "description": "Diagnose latency degradation (delay, QoT, BER) of an existing connection between two nodes "
                   "using telemetry and the digital twin, and reconfigure it over a new route.",
    "defaults": {"bitrate_gbps": 1},
    "steps": [
        {"intent": "retrieve end-to-end latency and QoT telemetry from the digital twin to diagnose degradation",
         "tool": "get_e2e_telemetry", "args": ["src_node", "dst_node"]},
        {"intent": "compute the shortest route between source and destination nodes for a candidate connection",
         "tool": "compute_route", "args": ["src_node", "dst_node"]},
        {"intent": "provision and bring up a new IP connection on the SDN-controlled network",
         "tool": "establish_ip_connection", "args": ["src_node", "dst_node", "bitrate_gbps"]},
    ],
}

P2MP_WORKFLOW = {
    "name": "allocate_p2mp_subcarriers",
    "description": "Allocate or assign digital subcarriers (capacity) to a leaf node (ONU) of the optical "
                   "point-to-multipoint P2MP access tree.",
    "defaults": {},
    "steps": [{"intent": "allocate contiguous digital subcarriers to a leaf node in the P2MP access tree",
               "tool": "allocate_subcarriers", "args": ["leaf_node", "num_sc"]}],
}


def _register_tool(mcp, fn):
    try:
        mcp.remove_tool(fn.__name__)      # re-ejecución idempotente de la celda
    except Exception:
        pass
    mcp.tool(fn)


def install(*, mcp, kb, planner, orchestrator, ip_agent, inventory, llm=None):
    """Registra herramientas MCP, workflows, el agente óptico y el gemelo digital."""
    p2mp = P2MPSubcarrierManager()
    twin = OCATADigitalTwinMock(inventory)

    def allocate_subcarriers(leaf_node: str, num_sc: int) -> dict:
        """Allocate contiguous 25 Gb/s digital subcarriers (DSCM) to a leaf node of the optical
        point-to-multipoint (P2MP) access tree without spectral overlap."""
        return p2mp.allocate_subcarriers(leaf_node, num_sc)

    def get_e2e_telemetry(src_node: str, dst_node: str) -> dict:
        """Get end-to-end telemetry of a provisioned connection from the OCATA digital twin:
        latency versus d_max, pre-FEC BER and QoT, to diagnose latency degradation."""
        return twin.estimate(src_node, dst_node)

    _register_tool(mcp, allocate_subcarriers)
    _register_tool(mcp, get_e2e_telemetry)
    kb.add_workflow(P2MP_WORKFLOW)
    kb.add_workflow(DIAGNOSE_WORKFLOW)
    planner.known_nodes |= set(P2MP_LEAVES)

    optical_agent = OpticalP2MPAgent(llm=llm, label="OpticalAgent")
    optical_agent.base_url = wc.start_server("optical_agent", a2a_agents.create_agent_app(optical_agent),
                                             "optical_agent")
    orchestrator.register_agent(optical_agent.base_url, label="OpticalAgent")

    ip_agent.allowed_tools = set(ip_agent.allowed_tools) | {"get_e2e_telemetry"}
    if not any(s["id"] == "latency-diagnosis" for s in ip_agent.skills):
        ip_agent.skills = list(ip_agent.skills) + [{
            "id": "latency-diagnosis", "name": "Latency degradation diagnosis",
            "description": "Diagnose latency/QoT degradation with digital-twin telemetry and reconfigure the connection.",
            "tags": ["telemetry", "digital-twin", "tool:get_e2e_telemetry", "tool:compute_route",
                     "tool:establish_ip_connection"],
            "examples": ["Diagnose the latency degradation between H1 and H2 and reconfigure the connection"],
            "inputModes": ["application/json"], "outputModes": ["application/json"]}]
    orchestrator.register_agent(ip_agent.base_url, label="Agent")
    return {"p2mp": p2mp, "optical_agent": optical_agent, "twin": twin}


In [ ]:
%%writefile workshop_dashboard.py
"""
workshop_dashboard.py
=====================
Visualización interactiva final del workshop (Módulo 8).

Ejecuta un escenario REAL sobre la arquitectura levantada en el notebook
(Orquestador -> LLM/ChromaDB -> Agente A2A -> MCP -> PCE/SDN -> OVS) y permite
recorrer, mensaje a mensaje, lo que ocurrió:

  * Arquitectura animada: qué componente habla con cuál en cada instante.
  * Captura estilo Wireshark sincronizada con la animación.
  * Plano de datos: ruta calculada, conexión provisionada, estado de enlaces,
    tablas de flujo OpenFlow por switch y ping antes/después.
  * Razonamiento: similitud de ChromaDB, argumentos extraídos, plan, traza y
    selección del agente (Handoff A2A).
  * Espectro P2MP del Hub (Extensión 2).

Solo usa ipywidgets + HTML/SVG (sin JavaScript propio), por lo que funciona en
JupyterLab, VS Code y Colab. Sin ipywidgets muestra una vista estática.
"""
import copy
import html
import json
import math
import time
from contextlib import contextmanager

import requests

import llm_planner

try:
    import ipywidgets as widgets
    from IPython.display import HTML, display
    WIDGETS_AVAILABLE = True
except ImportError:  # pragma: no cover
    widgets = None
    WIDGETS_AVAILABLE = False
    from IPython.display import HTML, display

STYLE = """
<style>
.wsd{background:#0b1220;color:#e6edf7;font-family:'Segoe UI',Roboto,Arial,sans-serif;border-radius:10px;padding:10px 12px;font-size:13px;line-height:1.4}
.wsd h4{margin:2px 0 8px;color:#4fc3f7;font-size:15px}
.wsd .mut{color:#9fb0cc}.wsd .ok{color:#81c784}.wsd .bad{color:#e57373}.wsd .acc{color:#4fc3f7}.wsd .acc2{color:#ffb74d}
.wsd pre{background:#0a1322;border:1px solid #2a3a5c;border-radius:8px;padding:8px 10px;font-family:Consolas,Menlo,monospace;font-size:11.5px;color:#d6e4f5;white-space:pre-wrap;word-break:break-word;margin:4px 0;max-height:300px;overflow:auto}
.wsd table{border-collapse:collapse;width:100%;font-size:12px;color:#e6edf7}
.wsd th,.wsd td{border-bottom:1px solid #22314f;padding:3px 6px;text-align:left;vertical-align:top;color:#e6edf7}
.wsd b{color:#ffffff}
.wsd th{color:#4fc3f7;background:#0f1a2e;position:sticky;top:0}
.wsd tr.cur td{background:#123a52;color:#fff}
.wsd tr.past td{color:#c9d6ea}.wsd tr.fut td{color:#56698c}
.wsd .card{background:#121b2e;border:1px solid #2a3a5c;border-radius:8px;padding:8px 10px;margin:4px 0}
.wsd .grid2{display:grid;grid-template-columns:1fr 1fr;gap:10px}
.wsd .pill{display:inline-block;border:1px solid #2a3a5c;border-radius:999px;padding:1px 9px;margin:2px 4px 2px 0;font-size:12px;background:#18233b}
.wsd .bar{height:14px;border-radius:4px;background:#2a3a5c}
.wsd .bar>div{height:14px;border-radius:4px}
</style>
"""

# --------------------------------------------------------------------- geometría del diagrama
NODES = {
    "User":         (15, 25, 135, 54, "👤 Operador", "intención"),
    "Orchestrator": (205, 25, 175, 54, "🧭 Orquestador", "POST /prompt"),
    "LLM":          (450, 18, 230, 68, "🧠 LLM + ChromaDB", "POST /plan · Mistral 7B"),
    "Agent":        (205, 165, 175, 54, "🤖 IPConnectionAgent", "A2A /execute_plan"),
    "OpticalAgent": (205, 265, 175, 54, "🔆 OpticalP2MPAgent", "A2A /execute_plan"),
    "MCP Server":   (450, 205, 175, 54, "🧰 Servidor MCP", "tools/list · tools/call"),
    "PCE":          (750, 110, 200, 54, "📐 PCE + Inventario", "POST /compute_route"),
    "SDN":          (750, 215, 200, 54, "🎛️ Controlador SDN", "NBI /service/ip/*"),
    "TFS (dry)":    (750, 18, 200, 54, "☁️ ETSI TFS (dry-run)", "/tfs-api/…/services"),
    "OVS r1":       (575, 380, 80, 44, "OVS R1", ""),
    "OVS r2":       (705, 380, 80, 44, "OVS R2", ""),
    "OVS r3":       (835, 380, 80, 44, "OVS R3", ""),
}
PROTO_COLORS = {"req": "#4fc3f7", "resp": "#81c784", "MCP": "#ffb74d", "OpenFlow": "#f06292"}


def _esc(text):
    return html.escape(str(text))


def _center(name):
    x, y, w, h = NODES[name][:4]
    return x + w / 2, y + h / 2


def _clip(name, dx, dy):
    """Distancia desde el centro hasta el borde del rectángulo en la dirección (dx, dy)."""
    _, _, w, h = NODES[name][:4]
    tx = (w / 2) / abs(dx) if dx else math.inf
    ty = (h / 2) / abs(dy) if dy else math.inf
    return min(tx, ty)


def _arrow_coords(src, dst):
    (x1, y1), (x2, y2) = _center(src), _center(dst)
    dx, dy = x2 - x1, y2 - y1
    norm = math.hypot(dx, dy) or 1.0
    ux, uy = dx / norm, dy / norm
    off = 5 if src < dst else -5                     # separa request/response
    px, py = -uy * off, ux * off
    a = _clip(src, ux, uy) + 2
    b = _clip(dst, -ux, -uy) + 4
    return (x1 + ux * a + px, y1 + uy * a + py, x2 - ux * b + px, y2 - uy * b + py)


def frame_color(frame):
    if frame["protocol"] == "HTTP":
        return PROTO_COLORS["resp"] if frame["info"].startswith("HTTP/1.1") else PROTO_COLORS["req"]
    return PROTO_COLORS.get(frame["protocol"], "#9fb0cc")


def explain(frame):
    """(paso, explicación) en lenguaje claro para cada trama capturada."""
    s, d, info, proto = frame["src"], frame["dst"], frame["info"], frame["protocol"]
    payload = frame.get("payload")
    is_resp = info.startswith("HTTP/1.1")
    if proto == "OpenFlow":
        if "PACKET_IN" in info:
            return "", f"El switch {d if s == 'SDN' else s} no tiene regla para el paquete y lo sube al controlador (PACKET_IN)."
        if "DELETE" in info:
            return "6", "El controlador BORRA las reglas del servicio (FLOW_MOD DELETE por cookie): vuelve el aislamiento."
        return "6", f"El controlador Ryu programa {d} con un OFPT_FLOW_MOD (OpenFlow 1.3, interfaz Sur)."
    if proto == "MCP":
        if "initialize" in info:
            return "MCP", f"{s} abre una sesión MCP con el servidor de herramientas."
        if "tools/list" in info:
            return "MCP", "El agente descubre las herramientas: nombre, descripción y JSON Schema de argumentos."
        if "result:" in info and "tools" in info:
            return "MCP", "El servidor MCP responde con el catálogo de herramientas disponibles."
        if "tools/call" in info:
            return "MCP", f"El agente invoca una herramienta determinista: {info.split('name=')[-1]} con {json.dumps(payload)}."
        return "MCP", "El servidor MCP devuelve el resultado determinista de la herramienta."
    if s == "User" and "/prompt" in info:
        return "1", f"El operador envía su intención en lenguaje natural: {payload.get('prompt') if isinstance(payload, dict) else ''!r}."
    if s == "User" and "/approve" in info:
        decision = "APRUEBA" if isinstance(payload, dict) and payload.get("approved") else "RECHAZA"
        return "HITL", f"El operador {decision} el paso pendiente (Human-in-the-Loop)."
    if s == "Orchestrator" and d == "LLM":
        return "2", "El Orquestador pide un plan: ChromaDB recupera el workflow más similar y el LLM extrae los argumentos."
    if s == "LLM" and is_resp:
        return "2", "El servicio LLM devuelve el plan estructurado {steps, plans} (Figura 5, mensaje 2)."
    if s == "Orchestrator" and "/.well-known" in info:
        return "A2A", f"El Orquestador lee la Agent Card de {d} (descubrimiento A2A)."
    if s == "Orchestrator" and "/execute_plan" in info:
        return "3", f"Handoff A2A: el Orquestador delega el plan a {d}, cuyas skills cubren las herramientas pedidas."
    if s == "Orchestrator" and "/a2a" in info:
        return "HITL", f"El Orquestador reenvía la decisión al agente: JSON-RPC SendMessage sobre la misma taskId."
    if s == "Agent" and d == "PCE":
        return "4", "Vía MCP, la herramienta compute_route consulta al PCE (cálculo determinista de ruta)."
    if s == "PCE" and is_resp:
        path = payload.get("path") if isinstance(payload, dict) else None
        return "4", f"El PCE responde la ruta: {' → '.join(path) if path else payload}."
    if d == "SDN" and "/service/ip/create" in info:
        return "5", "Vía MCP, establish_ip_connection invoca la NBI del controlador (POST /service/ip/create)."
    if d == "SDN" and "/service/ip/delete" in info:
        return "5", "Vía MCP, remove_ip_connection invoca la NBI del controlador (DELETE /service/ip/delete)."
    if d == "SDN" and "/service/ip/status" in info:
        return "5", "Vía MCP, get_connection_status consulta la NBI del controlador (GET /service/ip/status)."
    if s == "SDN" and is_resp:
        return "5", "El controlador SDN confirma la operación (HTTP 200 OK)."
    if d.startswith("TFS"):
        return "5", "La misma herramienta MCP genera el descriptor para ETSI TeraFlowSDN (modo dry-run)."
    if d == "Orchestrator" and is_resp:
        state = (payload or {}).get("status", {}).get("state", "") if isinstance(payload, dict) else ""
        if isinstance(payload, dict) and "task" in (payload.get("result") or {}):
            state = payload["result"]["task"]["status"]["state"]
        if state == "TASK_STATE_INPUT_REQUIRED":
            return "7", f"{s} pausa la Task en {state}: espera la aprobación del operador (HITL)."
        return "7", f"{s} devuelve la Task A2A en estado {state or 'final'} con la traza como artifact."
    if s == "Orchestrator" and d == "User" and is_resp:
        if isinstance(payload, dict) and payload.get("status") == "TASK_STATE_INPUT_REQUIRED":
            return "HITL", "El Orquestador muestra al operador la ruta propuesta y le pide aprobación."
        return "8", "El operador recibe la respuesta final: {\"trace\": [...]} (Figura 5, mensaje 8)."
    return "", info


# ===================================================================== dashboard
class WorkshopDashboard:
    def __init__(self, *, wc, orchestrator, orch_url, sdn_url, planner, kb, inventory, ip_agent,
                 prompt, ping_fn, net=None, optical_agent=None, p2mp=None, twin=None,
                 simulation_mode=True):
        self.wc, self.orch, self.orch_url, self.sdn_url = wc, orchestrator, orch_url, sdn_url
        self.planner, self.kb, self.inventory, self.ip_agent = planner, kb, inventory, ip_agent
        self.prompt, self.ping_fn, self.net = prompt, ping_fn, net
        self.optical_agent, self.p2mp, self.twin = optical_agent, p2mp, twin
        self.simulation_mode = simulation_mode
        self.real_llm = planner.llm
        self._no_llm = llm_planner.LocalLLM(enabled=False)
        self.result = None
        self.scenarios = self._build_scenarios()

    # ------------------------------------------------------------------ escenarios
    def _conn_exists(self):
        st = requests.get(f"{self.sdn_url}/service/ip/status", timeout=10).json()
        return "H1-H2" in st.get("connections", {})

    def _remove_conn(self):
        if self._conn_exists():
            requests.delete(f"{self.sdn_url}/service/ip/delete", params={"connection_id": "H1-H2"}, timeout=10)

    def _ensure_conn(self):
        requests.post(f"{self.sdn_url}/service/ip/create", timeout=10, json={
            "src_node": "H1", "dst_node": "H2", "bitrate_gbps": 1, "path": ["H1", "R1", "R2", "R3", "H2"]})

    def _degrade(self):
        self._ensure_conn()
        self.twin.inject_degradation("R2", "R3", extra_ms=2.0, ber=6e-3)
        self.inventory.set_link_state("R1", "R3", "up")

    def _undegrade(self):
        self.twin.clear()
        self.inventory.set_link_state("R1", "R3", "standby")

    def _build_scenarios(self):
        sc = {
            "paper": {"label": "① Demo del paper: conectar H1 → H2 con 1 Gb/s", "prompt": self.prompt,
                      "setup": self._remove_conn,
                      "about": "Reproduce las Figuras 3, 4 y 5: del aislamiento a la conectividad."},
            "hitl": {"label": "② Conectar H1 → H2 con aprobación humana (HITL)", "prompt": self.prompt,
                     "setup": self._remove_conn, "hitl": True,
                     "about": "La Task A2A se pausa en INPUT_REQUIRED tras el PCE; usted decide si se programa la red."},
            "status": {"label": "③ Consultar el estado de las conexiones",
                       "prompt": "Show me the status of the connections",
                       "about": "Otro workflow de ChromaDB: solo lectura de la NBI (GET /service/ip/status)."},
            "route": {"label": "④ Solo calcular la ruta H1 → H2 (what-if)",
                      "prompt": "Find the shortest path between H1 and H2",
                      "about": "Consulta al PCE sin tocar la red: no hay FLOW_MOD."},
            "remove": {"label": "⑤ Eliminar la conexión H1–H2 (vuelve el aislamiento)",
                       "prompt": "Remove the IP connection between H1 and H2", "setup": self._ensure_conn,
                       "about": "FLOW_MOD DELETE por cookie: el ping vuelve a fallar."},
        }
        if self.optical_agent is not None and self.p2mp is not None:
            sc["p2mp"] = {"label": "⑥ Ext. 2 · Asignar 3 subportadoras a ONU2 (agente óptico P2MP)",
                          "prompt": "Allocate 3 subcarriers to leaf ONU2 in the P2MP access tree",
                          "about": "El Orquestador hace Handoff al OpticalP2MPAgent por su Agent Card."}
        if self.twin is not None:
            sc["degradation"] = {"label": "⑦ Ext. 3 · Degradación en R2–R3 → diagnóstico y re-enrutamiento",
                                 "prompt": "Diagnose the latency degradation between H1 and H2 and reconfigure the connection",
                                 "setup": self._degrade, "teardown": self._undegrade,
                                 "about": "El gemelo digital detecta la degradación; el PCE evita R2–R3 y usa el bypass óptico."}
        sc["custom"] = {"label": "✍️ Prompt libre (escriba su propia intención)", "prompt": None,
                        "about": "Pruebe sus propias frases: ¿qué workflow recupera ChromaDB?"}
        return sc

    @contextmanager
    def _llm_mode(self, use_llm):
        agents = [a for a in (self.ip_agent, self.optical_agent) if a is not None]
        saved = (self.planner.llm, [(a, a.llm) for a in agents])
        if not (use_llm and self.real_llm.available):
            self.planner.llm = self._no_llm
            for a in agents:
                a.llm = None
        try:
            yield
        finally:
            self.planner.llm = saved[0]
            for a, l in saved[1]:
                a.llm = l

    # ------------------------------------------------------------------ ejecución
    def _snapshot_links(self):
        snap = {}
        for u, v, d in self.inventory.graph.edges(data=True):
            snap[frozenset((u, v))] = {"u": u, "v": v, **{k: d[k] for k in ("layer", "state", "capacity_gbps", "used_gbps")}}
        return snap

    def run(self, key="paper", custom_prompt="", use_llm=False, approve=True):
        sc = self.scenarios[key]
        prompt = (custom_prompt or "").strip() if key == "custom" else sc["prompt"]
        if not prompt:
            raise ValueError("Escriba un prompt")
        if sc.get("setup"):
            sc["setup"]()
        ping_before = self.ping_fn()
        error, response, hitl_info = None, None, None
        links_during = None
        t0 = time.time()
        with self._llm_mode(use_llm):
            self.wc.CAPTURE.reset()
            try:
                if sc.get("hitl"):
                    self.ip_agent.hitl = True
                response = self.wc.http_call("User", "Orchestrator", "POST", f"{self.orch_url}/prompt",
                                             {"prompt": prompt})
                if isinstance(response, dict) and response.get("status") == "TASK_STATE_INPUT_REQUIRED":
                    hitl_info = response
                    response = self.wc.http_call("User", "Orchestrator", "POST", f"{self.orch_url}/approve",
                                                 {"task_id": response["task_id"], "approved": bool(approve)})
            except Exception as exc:
                error = getattr(exc, "detail", None) or str(exc)
            finally:
                self.ip_agent.hitl = False
                links_during = self._snapshot_links()
                frames = copy.deepcopy(self.wc.CAPTURE.frames)
                if sc.get("teardown"):
                    sc["teardown"]()
        elapsed = time.time() - t0
        ping_after = self.ping_fn()
        status = requests.get(f"{self.sdn_url}/service/ip/status", timeout=10).json()
        trace = response.get("trace", []) if isinstance(response, dict) else []
        computed = next((s["result"]["path"] for s in trace
                         if isinstance(s.get("result"), dict) and "path" in s["result"] and s["tool"].startswith("compute_route")), None)
        if computed is None and hitl_info:
            computed = (hitl_info.get("details", {}).get("last_result") or {}).get("path")
        flows = {}
        if self.net is not None:
            for sid, sw in self.net.switches.items():
                flows[sid] = sw.dump_flows()
        self.result = {
            "key": key, "label": sc["label"], "about": sc.get("about", ""), "prompt": prompt,
            "use_llm": bool(use_llm and self.real_llm.available), "elapsed": elapsed,
            "frames": frames, "response": response, "error": error, "hitl": hitl_info, "approve": approve,
            "plan": copy.deepcopy(self.orch.last_plan), "handoff": self.orch.last_handoff,
            "retrieval": self.kb.search(prompt, k=self.kb.collection.count()),
            "extractor": self.planner.last_backend, "raw": copy.deepcopy(self.planner.last_raw),
            "ping_before": ping_before, "ping_after": ping_after, "sdn_status": status,
            "links": links_during, "computed_path": computed, "flows": flows,
            "p2mp": list(self.p2mp.slots) if self.p2mp is not None else None,
            "agents": {n: sorted(a["tools"]) for n, a in self.orch.agents.items()},
        }
        return self.result

    def visible_frames(self, show_mcp=True, show_of=True):
        if not self.result:
            return []
        out = []
        for f in self.result["frames"]:
            if f["protocol"] == "MCP" and not show_mcp:
                continue
            if f["protocol"] == "OpenFlow" and not show_of:
                continue
            out.append(f)
        return out

    # ------------------------------------------------------------------ renderizado
    def render_summary(self):
        r = self.result
        if not r:
            return STYLE + "<div class='wsd'>Sin ejecución todavía.</div>"
        counts = {}
        for f in r["frames"]:
            counts[f["protocol"]] = counts.get(f["protocol"], 0) + 1
        ok_before = "64 bytes" in r["ping_before"]
        ok_after = "64 bytes" in r["ping_after"]
        status = (f"<span class='bad'>❌ Error: {_esc(r['error'])}</span>" if r["error"]
                  else "<span class='ok'>✅ Flujo completado</span>")
        if r["hitl"] and not r["approve"]:
            status = "<span class='acc2'>🛑 Operación rechazada por el operador (Task CANCELED)</span>"
        llm = "Mistral 7B real (Ollama)" if r["use_llm"] else "LLM emulado (rápido)"
        pills = "".join(f"<span class='pill'>{_esc(k)}: {v}</span>" for k, v in counts.items())
        return STYLE + f"""<div class='wsd'>
<h4>{_esc(r['label'])}</h4>
<div>🗣️ <b>Prompt:</b> <code>{_esc(r['prompt'])}</code></div>
<div class='mut'>{_esc(r['about'])}</div>
<div style='margin-top:6px'>{status} · ⏱️ {r['elapsed']:.1f} s · 🧠 {llm} (extractor: {_esc(r['extractor'])})
 · 🤝 Handoff → <b>{_esc(r['handoff'])}</b></div>
<div style='margin-top:4px'>Plano de datos <code>h1 ping h2</code>: antes {'<span class=ok>✅ alcanzable</span>' if ok_before else '<span class=bad>⛔ inalcanzable</span>'}
 → después {'<span class=ok>✅ alcanzable</span>' if ok_after else '<span class=bad>⛔ inalcanzable</span>'}</div>
<div style='margin-top:4px'>Mensajes capturados: {pills}</div></div>"""

    def render_architecture(self, frames, idx):
        cur = frames[idx] if frames else None
        parts = [f"<svg viewBox='0 0 970 440' width='100%' style='background:#0b1220;border-radius:10px'>",
                 "<defs>" + "".join(
                     f"<marker id='m{k}' markerWidth='9' markerHeight='9' refX='8' refY='4.5' orient='auto'>"
                     f"<path d='M0,0 L9,4.5 L0,9 z' fill='{c}'/></marker>" for k, c in
                     [("req", PROTO_COLORS["req"]), ("resp", PROTO_COLORS["resp"]), ("MCP", PROTO_COLORS["MCP"]),
                      ("OpenFlow", PROTO_COLORS["OpenFlow"]), ("past", "#3a4c70")]) + "</defs>"]
        parts.append("<rect x='545' y='355' width='415' height='80' rx='10' fill='#122318' stroke='#2e5a3a'/>"
                     "<text x='555' y='372' fill='#81c784' font-size='12'>Plano de datos (Mininet / OVS)</text>")
        active = {cur["src"], cur["dst"]} if cur else set()
        used = set()
        for f in frames[: idx + 1]:
            used |= {f["src"], f["dst"]}
        # aristas pasadas (tenues)
        seen = set()
        for f in frames[:idx]:
            if f["src"] in NODES and f["dst"] in NODES and (f["src"], f["dst"]) not in seen:
                seen.add((f["src"], f["dst"]))
                x1, y1, x2, y2 = _arrow_coords(f["src"], f["dst"])
                parts.append(f"<line x1='{x1:.0f}' y1='{y1:.0f}' x2='{x2:.0f}' y2='{y2:.0f}' stroke='#3a4c70' "
                             f"stroke-width='1.5' marker-end='url(#mpast)'/>")
        for name, (x, y, w, h, title, sub) in NODES.items():
            if name == "OpticalAgent" and self.optical_agent is None:
                continue
            if name == "TFS (dry)" and name not in used:
                continue
            stroke = "#ffffff" if name in active else ("#4fc3f7" if name in used else "#2a3a5c")
            fill = "#17324a" if name in active else "#121b2e"
            sw = 3 if name in active else 1.3
            parts.append(f"<rect x='{x}' y='{y}' width='{w}' height='{h}' rx='9' fill='{fill}' stroke='{stroke}' stroke-width='{sw}'/>")
            parts.append(f"<text x='{x + w / 2}' y='{y + (h / 2 + 4 if not sub else 23)}' fill='#fff' font-size='13' "
                         f"font-weight='700' text-anchor='middle'>{_esc(title)}</text>")
            if sub:
                parts.append(f"<text x='{x + w / 2}' y='{y + 41}' fill='#9fb0cc' font-size='11' text-anchor='middle'>{_esc(sub)}</text>")
        parts.append("<g font-size='11' fill='#9fb0cc'><text x='575' y='432'>H1 ─ R1 ─ R2 ─ R3 ─ H2</text></g>")
        if cur and cur["src"] in NODES and cur["dst"] in NODES:
            x1, y1, x2, y2 = _arrow_coords(cur["src"], cur["dst"])
            color = frame_color(cur)
            mk = "OpenFlow" if cur["protocol"] == "OpenFlow" else ("MCP" if cur["protocol"] == "MCP" else
                                                                  ("resp" if cur["info"].startswith("HTTP/1.1") else "req"))
            parts.append(f"<line x1='{x1:.0f}' y1='{y1:.0f}' x2='{x2:.0f}' y2='{y2:.0f}' stroke='{color}' "
                         f"stroke-width='4.5' marker-end='url(#m{mk})'/>")
            step, _ = explain(cur)
            if step:
                mx, my = (x1 + x2) / 2, (y1 + y2) / 2
                bw = 22 if len(step) <= 2 else 12 + 8 * len(step)
                parts.append(f"<rect x='{mx - bw / 2:.0f}' y='{my - 11:.0f}' width='{bw}' height='22' rx='11' fill='{color}'/>"
                             f"<text x='{mx:.0f}' y='{my + 4:.0f}' fill='#04121f' font-size='11' font-weight='800' "
                             f"text-anchor='middle'>{_esc(step)}</text>")
        parts.append("<g font-size='11'>"
                     f"<text x='15' y='395' fill='{PROTO_COLORS['req']}'>━ HTTP request</text>"
                     f"<text x='15' y='412' fill='{PROTO_COLORS['resp']}'>━ HTTP response</text>"
                     f"<text x='150' y='395' fill='{PROTO_COLORS['MCP']}'>━ MCP (JSON-RPC)</text>"
                     f"<text x='150' y='412' fill='{PROTO_COLORS['OpenFlow']}'>━ OpenFlow 1.3</text>"
                     "<text x='15' y='429' fill='#6b7f9e'>etiqueta sobre la flecha = paso de la Figura 2 del paper</text></g>")
        parts.append("</svg>")
        return "".join(parts)

    def render_detail(self, frames, idx):
        if not frames:
            return STYLE + "<div class='wsd'>No hay mensajes con los filtros actuales.</div>"
        f = frames[idx]
        step, text = explain(f)
        payload = f.get("payload")
        body = ""
        if payload not in (None, ""):
            raw = json.dumps(payload, indent=2, ensure_ascii=False) if not isinstance(payload, str) else payload
            if len(raw) > 2500:
                raw = raw[:2500] + "\n… (truncado)"
            body = f"<pre>{_esc(raw)}</pre>"
        return STYLE + f"""<div class='wsd'>
<div><span class='pill' style='background:{frame_color(f)};color:#04121f;font-weight:700'>{_esc(f['protocol'])}</span>
<b>Mensaje {idx + 1} / {len(frames)}</b> · t = {f['time']:.4f} s · {('Paso ' + _esc(step)) if step else ''}</div>
<div style='margin:6px 0'><b>{_esc(f['src'])}</b> → <b>{_esc(f['dst'])}</b> &nbsp;<code>{_esc(f['info'])}</code></div>
<div class='card'>💡 {_esc(text)}</div>{body}</div>"""

    def render_table(self, frames, idx):
        rows = []
        for i, f in enumerate(frames):
            cls = "cur" if i == idx else ("past" if i < idx else "fut")
            step, _ = explain(f)
            rows.append(f"<tr class='{cls}'><td>{i + 1}</td><td>{f['time']:.4f}</td><td>{_esc(f['src'])}</td>"
                        f"<td>{_esc(f['dst'])}</td><td>{_esc(f['protocol'])}</td><td>{_esc(step)}</td>"
                        f"<td>{_esc(f['info'][:95])}</td></tr>")
        return STYLE + ("<div class='wsd'><div style='max-height:250px;overflow:auto'><table><tr><th>No.</th><th>Time</th>"
                        "<th>Source</th><th>Destination</th><th>Proto</th><th>Paso</th><th>Info</th></tr>"
                        + "".join(rows) + "</table></div></div>")

    def render_topology(self, switch="R2"):
        r = self.result
        if not r:
            return ""
        pos = {"H1": (60, 95), "R1": (230, 95), "R2": (420, 95), "R3": (610, 95), "H2": (780, 95)}
        conn = r["sdn_status"].get("connections", {}).get("H1-H2")
        conn_path = conn["path"] if conn else []
        conn_edges = {frozenset(e) for e in zip(conn_path, conn_path[1:])}
        comp = r["computed_path"] or []
        comp_edges = {frozenset(e) for e in zip(comp, comp[1:])}
        svg = ["<svg viewBox='0 0 840 250' width='100%' style='background:#0b1220;border-radius:10px'>"]
        for key, l in r["links"].items():
            (x1, y1), (x2, y2) = pos[l["u"]], pos[l["v"]]
            color = {"up": "#c9d6ea", "standby": "#ba68c8", "degraded": "#e57373"}.get(l["state"], "#9fb0cc")
            dash = "8 6" if l["state"] != "up" else ""
            residual = l["capacity_gbps"] - l["used_gbps"]
            if l["layer"] == "optical":
                d = f"M{x1},{y1 + 20} Q{(x1 + x2) / 2},{y1 + 150} {x2},{y2 + 20}"
                lx, ly = (x1 + x2) / 2, y1 + 100
            else:
                d = f"M{x1},{y1} L{x2},{y2}"
                lx, ly = (x1 + x2) / 2, y1 - 14
            if key in conn_edges:
                svg.append(f"<path d='{d}' fill='none' stroke='#4fc3f7' stroke-width='12' stroke-opacity='0.35'/>")
            if key in comp_edges:
                svg.append(f"<path d='{d}' fill='none' stroke='#ffb74d' stroke-width='3' stroke-dasharray='3 5' "
                           f"transform='translate(0,6)'/>")
            svg.append(f"<path d='{d}' fill='none' stroke='{color}' stroke-width='2.5' stroke-dasharray='{dash}'/>")
            svg.append(f"<text x='{lx}' y='{ly}' fill='{color}' font-size='11' text-anchor='middle'>"
                       f"{l['state']} · {residual:g}/{l['capacity_gbps']:g} Gb/s libres</text>")
        for n, (x, y) in pos.items():
            if n.startswith("H"):
                svg.append(f"<circle cx='{x}' cy='{y}' r='24' fill='#9ecae1'/><text x='{x}' y='{y + 5}' "
                           f"fill='#04121f' font-weight='800' text-anchor='middle'>{n}</text>")
            else:
                stroke = "#ffffff" if n == switch else "#fdae6b"
                svg.append(f"<rect x='{x - 28}' y='{y - 20}' width='56' height='40' rx='6' fill='#fdae6b' "
                           f"stroke='{stroke}' stroke-width='3'/><text x='{x}' y='{y + 5}' fill='#04121f' "
                           f"font-weight='800' text-anchor='middle'>{n}</text>")
        svg.append("<g font-size='11'><text x='15' y='215' fill='#4fc3f7'>▬ conexión provisionada en el SDN</text>"
                   "<text x='15' y='232' fill='#ffb74d'>┅ ruta calculada por el PCE en esta ejecución</text>"
                   "<text x='330' y='215' fill='#ba68c8'>┅ bypass óptico en standby</text>"
                   "<text x='330' y='232' fill='#e57373'>┅ enlace degradado (gemelo digital)</text></g></svg>")
        conn_txt = (f"<b>H1-H2</b> UP por {' → '.join(conn_path)} ({conn['bitrate_gbps']:g} Gb/s, {conn['flow_mods']} FLOW_MOD)"
                    if conn else "<span class='bad'>sin conexión H1-H2: el aislamiento está activo</span>")
        flows = r["flows"].get(switch)
        flows_html = (f"<pre>$ ovs-ofctl -O OpenFlow13 dump-flows {switch.lower()}\n{_esc(flows)}</pre>" if flows is not None
                      else "<div class='mut'>Modo real: use <code>dump_flows('R2')</code> o <code>sudo ovs-ofctl -O OpenFlow13 dump-flows r2</code>.</div>")
        return STYLE + f"""<div class='wsd'>{''.join(svg)}
<div class='card'>🎛️ Estado del controlador SDN: {conn_txt}</div>
<div class='grid2'><div><h4>mininet&gt; h1 ping h2 · ANTES</h4><pre>{_esc(r['ping_before'])}</pre></div>
<div><h4>mininet&gt; h1 ping h2 · DESPUÉS</h4><pre>{_esc(r['ping_after'])}</pre></div></div>
<h4>Tabla de flujos de {switch} (seleccione el switch arriba)</h4>{flows_html}</div>"""

    def render_reasoning(self):
        r = self.result
        if not r:
            return ""
        bars = []
        best = r["retrieval"][0]["name"] if r["retrieval"] else None
        for h in r["retrieval"]:
            pct = max(0.0, min(1.0, h["similarity"])) * 100
            color = "#4fc3f7" if h["name"] == best else "#3a4c70"
            bars.append(f"<tr><td style='width:34%'>{'◀ ' if h['name'] == best else ''}<b>{_esc(h['name'])}</b></td>"
                        f"<td><div class='bar'><div style='width:{pct:.0f}%;background:{color}'></div></div></td>"
                        f"<td style='width:60px'>{h['similarity']:.3f}</td></tr>")
        agents = []
        needed = {p["tool"] for p in (r["plan"] or {}).get("plans", [])}
        for name, tools in r["agents"].items():
            sel = name == r["handoff"]
            tl = " ".join(f"<span class='pill' style='{'border-color:#81c784;color:#81c784' if t in needed else ''}'>{_esc(t)}</span>"
                          for t in tools)
            agents.append(f"<div class='card' style='{'border-color:#4fc3f7' if sel else ''}'>"
                          f"{'🤝 <b>' if sel else ''}{_esc(name)}{'</b> ← Handoff' if sel else ''}<br>{tl}</div>")
        plan = json.dumps(r["plan"], indent=2, ensure_ascii=False) if r["plan"] else "—"
        trace = json.dumps(r["response"], indent=2, ensure_ascii=False) if r["response"] else _esc(r["error"])
        return STYLE + f"""<div class='wsd'>
<h4>1 · ChromaDB: similitud coseno entre el prompt y cada workflow (zero-shot)</h4><table>{''.join(bars)}</table>
<h4 style='margin-top:10px'>2 · LLM: argumentos extraídos ({_esc(r['extractor'])})</h4><pre>{_esc(json.dumps(r['raw'], ensure_ascii=False))}</pre>
<h4>3 · A2A: agentes registrados (verde = herramienta que el plan necesita)</h4>{''.join(agents)}
<div class='grid2'><div><h4>Plan · mensaje 2</h4><pre>{_esc(plan)}</pre></div>
<div><h4>Respuesta al operador · mensaje 8</h4><pre>{_esc(trace)}</pre></div></div></div>"""

    def render_p2mp(self):
        r = self.result
        if not r or r["p2mp"] is None:
            return STYLE + "<div class='wsd mut'>La Extensión 2 (P2MP) no está disponible en esta sesión.</div>"
        palette = ["#4fc3f7", "#ffb74d", "#81c784", "#ba68c8", "#f06292", "#ffd54f"]
        leaves = sorted({s for s in r["p2mp"] if s})
        color = {l: palette[i % len(palette)] for i, l in enumerate(leaves)}
        cells = "".join(f"<rect x='{10 + i * 50}' y='20' width='46' height='46' rx='5' fill='{color.get(s, '#2a3a5c')}'/>"
                        f"<text x='{33 + i * 50}' y='48' font-size='11' text-anchor='middle' fill='#04121f' font-weight='700'>{i}</text>"
                        for i, s in enumerate(r["p2mp"]))
        legend = "".join(f"<span class='pill' style='border-color:{color[l]};color:{color[l]}'>{_esc(l)}: "
                         f"{r['p2mp'].count(l)} SC = {25 * r['p2mp'].count(l)} Gb/s</span>" for l in leaves)
        used = sum(1 for s in r["p2mp"] if s)
        return STYLE + f"""<div class='wsd'><h4>Espectro del Hub P2MP: 16 subportadoras × 25 Gb/s</h4>
<svg viewBox='0 0 820 80' width='100%'>{cells}</svg>
<div>{legend or '<span class=mut>sin asignaciones</span>'}</div>
<div class='mut' style='margin-top:6px'>Ocupación del Hub: {used}/16 ({used * 25} de 400 Gb/s). Política First-Fit contigua,
sin solapamiento espectral; máx. 4 SC por hoja. Ejecute el escenario ⑥ varias veces o con otras hojas en «Prompt libre»
(p.ej. <code>Allocate 4 subcarriers to leaf ONU3 in the P2MP access tree</code>).</div></div>"""

    def render_static(self):
        frames = self.visible_frames()
        idx = len(frames) - 1 if frames else 0
        return (self.render_summary() + self.render_architecture(frames, idx) + self.render_table(frames, idx)
                + self.render_topology() + self.render_reasoning() + self.render_p2mp())

    # ------------------------------------------------------------------ interfaz ipywidgets
    def show(self, default="paper"):
        if not WIDGETS_AVAILABLE:
            self.run(default)
            display(HTML(self.render_static()))
            print("ℹ️ Instale ipywidgets para la versión interactiva: pip install ipywidgets")
            return
        W = widgets
        scenario = W.Dropdown(options=[(v["label"], k) for k, v in self.scenarios.items()], value=default,
                              description="Escenario:", layout=W.Layout(width="620px"),
                              style={"description_width": "80px"})
        custom = W.Text(placeholder="p.ej. Establish an IP connection from H1 to H2 with 2 Gbps",
                        description="Prompt:", layout=W.Layout(width="620px", display="none"),
                        style={"description_width": "80px"})
        use_llm = W.Checkbox(value=False, indent=False,
                             description=("Usar Mistral 7B real (Ollama) — más lento en CPU" if self.real_llm.available
                                          else "Mistral 7B no disponible: se usa el LLM emulado"),
                             disabled=not self.real_llm.available, layout=W.Layout(width="420px"))
        decision = W.ToggleButtons(options=[("✅ Aprobar", True), ("🛑 Rechazar", False)], value=True,
                                   description="Operador:", layout=W.Layout(display="none"))
        run_btn = W.Button(description="▶ Ejecutar escenario", button_style="primary", icon="play",
                           layout=W.Layout(width="200px"))
        status = W.HTML()
        show_mcp = W.Checkbox(value=True, description="Mostrar MCP", indent=False, layout=W.Layout(width="140px"))
        show_of = W.Checkbox(value=True, description="Mostrar OpenFlow", indent=False, layout=W.Layout(width="160px"))
        play = W.Play(value=0, min=0, max=0, step=1, interval=1100, description="Reproducir")
        slider = W.IntSlider(value=0, min=0, max=0, description="Mensaje:", continuous_update=True,
                             layout=W.Layout(width="520px"))
        W.jslink((play, "value"), (slider, "value"))
        summary_html, arch_html, detail_html, table_html = W.HTML(), W.HTML(), W.HTML(), W.HTML()
        switch_sel = W.ToggleButtons(options=["R1", "R2", "R3"], value="R2", description="Switch:")
        topo_html, reason_html, p2mp_html = W.HTML(), W.HTML(), W.HTML()

        state = {"frames": []}

        def refresh_frames(*_):
            state["frames"] = self.visible_frames(show_mcp.value, show_of.value)
            n = max(len(state["frames"]) - 1, 0)
            slider.max = play.max = n
            slider.value = play.value = 0
            refresh_frame()

        def refresh_frame(*_):
            frames, idx = state["frames"], min(slider.value, max(len(state["frames"]) - 1, 0))
            arch_html.value = self.render_architecture(frames, idx)
            detail_html.value = self.render_detail(frames, idx)
            table_html.value = self.render_table(frames, idx)

        def refresh_all():
            summary_html.value = self.render_summary()
            topo_html.value = self.render_topology(switch_sel.value)
            reason_html.value = self.render_reasoning()
            p2mp_html.value = self.render_p2mp()
            refresh_frames()

        def on_scenario(change):
            custom.layout.display = "" if change["new"] == "custom" else "none"
            decision.layout.display = "" if self.scenarios[change["new"]].get("hitl") else "none"

        def on_run(_):
            run_btn.disabled = True
            status.value = ("<span style='color:#ffb74d'>⏳ Ejecutando el flujo agéntico… "
                            + ("(Mistral en CPU puede tardar 1–2 min)" if use_llm.value else "") + "</span>")
            try:
                self.run(scenario.value, custom.value, use_llm.value, decision.value)
                status.value = "<span style='color:#81c784'>Listo. Use ▶ o el deslizador para recorrer los mensajes.</span>"
                refresh_all()
            except Exception as exc:
                status.value = f"<span style='color:#e57373'>Error: {_esc(exc)}</span>"
            finally:
                run_btn.disabled = False

        scenario.observe(on_scenario, names="value")
        run_btn.on_click(on_run)
        slider.observe(refresh_frame, names="value")
        show_mcp.observe(refresh_frames, names="value")
        show_of.observe(refresh_frames, names="value")
        switch_sel.observe(lambda c: setattr(topo_html, "value", self.render_topology(c["new"])), names="value")

        tab_flow = W.VBox([W.HBox([play, slider, show_mcp, show_of]), arch_html, detail_html, table_html])
        tab = W.Tab(children=[tab_flow, W.VBox([switch_sel, topo_html]), reason_html, p2mp_html])
        for i, t in enumerate(["🔁 Flujo de mensajes", "🔀 Plano de datos", "🧠 Razonamiento y A2A", "🌈 Espectro P2MP"]):
            tab.set_title(i, t)
        controls = W.VBox([scenario, custom, W.HBox([use_llm, decision]), W.HBox([run_btn, status])])
        display(W.VBox([W.HTML(STYLE + "<div class='wsd'><h4>🛰️ Visualización interactiva del sistema multi-agéntico</h4>"
                                       "Elija un escenario y pulse <b>Ejecutar</b>: se ejecuta de verdad sobre los servicios "
                                       "levantados en este notebook. Luego recorra los mensajes con ▶ o el deslizador.</div>"),
                        controls, summary_html, tab]))
        on_scenario({"new": scenario.value})
        on_run(None)


In [ ]:
# ▶️ Cargamos las extensiones de acceso (mismo patrón: herramienta MCP + workflow + agente A2A)
access_extensions = load_module("access_extensions")
ext = access_extensions.install(mcp=mcp, kb=kb, planner=planner, orchestrator=orchestrator, ip_agent=ip_agent,
                                inventory=inventory, llm=llm if AGENT_LLM_TOOL_MAPPING else None)
print("Agentes A2A registrados en el Orquestador:")
for name, a in orchestrator.agents.items():
    print(f"  🤖 {name:<18} herramientas: {sorted(a['tools'])}")
print(f"Workflows en ChromaDB: {kb.collection.count()}")

In [ ]:
# ▶️ 🛰️ CONSOLA INTERACTIVA: elija un escenario, pulse «Ejecutar» y recorra los mensajes con ▶
workshop_dashboard = load_module("workshop_dashboard")

def ping_text(count=3):
    if SIMULATION_MODE:
        return net.ping("H1", "H2", count)
    return requests.get(f"{wc.url('mininet_ctl')}/ping", params={"src": "h1", "dst": "h2", "count": count},
                        timeout=60).json()["output"]

dashboard = workshop_dashboard.WorkshopDashboard(
    wc=wc, orchestrator=orchestrator, orch_url=ORCH_URL, sdn_url=SDN_URL, planner=planner, kb=kb,
    inventory=inventory, ip_agent=ip_agent, prompt=PROMPT, ping_fn=ping_text,
    net=net if SIMULATION_MODE else None, optical_agent=ext["optical_agent"], p2mp=ext["p2mp"],
    twin=ext["twin"], simulation_mode=SIMULATION_MODE)
dashboard.show()

---
## 🏁 Cierre · ⏱️ 5 min
**Tres ideas para llevarse**
1. **El LLM entiende, el código ejecuta:** el modelo local solo interpreta la intención; rutas y reglas salen de herramientas **deterministas y auditables** (MCP).
2. **Integración sin romper nada:** se extendió la **NBI** del controlador (Ryu / TeraFlowSDN); la SBI OpenFlow hacia los equipos no cambió.
3. **Escalable por especialistas:** agregar una capacidad de acceso (P2MP, gemelo digital) = *herramienta MCP + workflow + agente A2A*, sin reentrenar nada.

**Para profundizar fuera de clase:** QoS en el PCE con el Algoritmo 1 de la Ref. [1] ($e.cap_{res} < \tau \cdot x^{max}$) · backend real de TeraFlowSDN (`wc.save_config(SDN_BACKEND="tfs", TFS_DRY_RUN=False)`) · modo real con Mininet + Ryu en Linux (`SIMULATION_MODE = False`) · embeddings neuronales (`ollama pull nomic-embed-text`).

> Al terminar puede detener todos los servicios con `wc.stop_all_servers()`.

**Referencias:** P. González, L. Velasco, IEEE 2026 · [1] P. González *et al.*, JOCN 17, A247–A258, 2025 · [2] H. Shakespear-Miles *et al.*, JOCN 16, 565–576, 2024 · [3] M. Devigili *et al.*, JOCN 16, 221–232, 2024 · a2a-protocol.org · modelcontextprotocol.io · mininet.org · ryu-sdn.org · tfs.etsi.org · docs.trychroma.com · ollama.com